### Model training - train_model.py

In [ ]:
"""
train_model.py
--------------
Trains a Random Forest classifier on the Car Insurance Claim dataset,
evaluates it, and persists the model + label encoders to disk so the
Flask app can load them without re-training.

Run once before starting the web application:
    python train_model.py
"""

import os
import pickle
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
import matplotlib
matplotlib.use("Agg")          # headless backend — no display needed
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ── 1. Load data ──────────────────────────────────────────────────────────────
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_PATH = os.path.join(BASE_DIR, "Car_Insurance_Claim.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['OUTCOME'].value_counts()}\n")

# ── 2. Pre-process ────────────────────────────────────────────────────────────
# Drop the ID column — it carries no predictive signal
df.drop(columns=["ID"], inplace=True)

# Fill numeric nulls with median; categorical nulls with mode
for col in df.columns:
    if df[col].dtype == "object":
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

# Encode categorical columns and save encoders for inference
CATEGORICAL_COLS = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE",
    "EDUCATION", "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE",
]

encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# ── 3. Feature / target split ─────────────────────────────────────────────────
X = df.drop(columns=["OUTCOME"])
y = df["OUTCOME"].astype(int)

FEATURE_NAMES = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── 4. Train multiple models and pick the best ────────────────────────────────
candidates = {
    "Random Forest":        RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1),
    "Gradient Boosting":    GradientBoostingClassifier(n_estimators=150, random_state=42),
    "Logistic Regression":  LogisticRegression(max_iter=1000, random_state=42),
}

results = {}
for name, clf in candidates.items():
    cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring="roc_auc")
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    auc   = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
    results[name] = {"model": clf, "accuracy": acc, "auc": auc, "cv_mean": cv_scores.mean()}
    print(f"{name:25s}  Accuracy={acc:.4f}  AUC={auc:.4f}  CV-AUC={cv_scores.mean():.4f}")

best_name = max(results, key=lambda k: results[k]["auc"])
best_model = results[best_name]["model"]
print(f"\nBest model: {best_name}  (AUC={results[best_name]['auc']:.4f})\n")

# ── 5. Detailed evaluation ────────────────────────────────────────────────────
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No Claim", "Claim"]))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No Claim", "Claim"],
            yticklabels=["No Claim", "Claim"])
axes[0].set_title(f"Confusion Matrix — {best_name}")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# Feature importance (available for tree-based models)
if hasattr(best_model, "feature_importances_"):
    fi = pd.Series(best_model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=True)
    fi.plot(kind="barh", ax=axes[1], color="steelblue")
    axes[1].set_title("Feature Importances")
    axes[1].set_xlabel("Importance")

plt.tight_layout()
plot_path = os.path.join(BASE_DIR, "static", "model_evaluation.png")
plt.savefig(plot_path, dpi=120)
plt.close()
print(f"Evaluation plot saved -> {plot_path}")

# ── 6. Persist artefacts ──────────────────────────────────────────────────────
artefacts = {
    "model":         best_model,
    "encoders":      encoders,
    "feature_names": FEATURE_NAMES,
    "model_name":    best_name,
    "accuracy":      results[best_name]["accuracy"],
    "auc":           results[best_name]["auc"],
}

artefact_path = os.path.join(BASE_DIR, "model.pkl")
with open(artefact_path, "wb") as f:
    pickle.dump(artefacts, f)

print(f"Model artefacts saved -> {artefact_path}")
print("\nTraining complete. You can now run:  python app.py")


### Backend - Flask Rest API - app.py

In [ ]:
"""
app.py  —  Flask REST API backend
==================================
Exposes two endpoints consumed by the Streamlit frontend.

  POST /predict   – accepts JSON, returns prediction + probability
  GET  /stats     – returns dataset & model summary statistics

Run:
    python app.py
    # server starts on http://127.0.0.1:5000
"""

import os
import pickle

import numpy as np
import pandas as pd
from flask import Flask, request, jsonify
from flask_cors import CORS

# ── App ────────────────────────────────────────────────────────────────────────
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
app = Flask(__name__)
CORS(app)   # allow Streamlit (different port) to call this API

# ── Load model artefacts ───────────────────────────────────────────────────────
ARTEFACT_PATH = os.path.join(BASE_DIR, "model.pkl")

if not os.path.exists(ARTEFACT_PATH):
    raise FileNotFoundError("model.pkl not found. Run `python train_model.py` first.")

with open(ARTEFACT_PATH, "rb") as f:
    _art = pickle.load(f)

MODEL         = _art["model"]
ENCODERS      = _art["encoders"]
FEATURE_NAMES = _art["feature_names"]
MODEL_NAME    = _art["model_name"]
MODEL_ACC     = _art["accuracy"]
MODEL_AUC     = _art["auc"]

# ── Dataset stats (built once at startup) ─────────────────────────────────────
_df = pd.read_csv(os.path.join(BASE_DIR, "Car_Insurance_Claim.csv"))

def _build_stats():
    df    = _df.copy()
    total = len(df)
    claim = int(df["OUTCOME"].sum())
    return {
        "total":              total,
        "claim_count":        claim,
        "no_claim_count":     total - claim,
        "claim_pct":          round(claim / total * 100, 1),
        "age_dist":           df["AGE"].value_counts().to_dict(),
        "income_dist":        df["INCOME"].value_counts().to_dict(),
        "vehicle_type_dist":  df["VEHICLE_TYPE"].value_counts().to_dict(),
        "gender_dist":        df["GENDER"].value_counts().to_dict(),
        "education_dist":     df["EDUCATION"].value_counts().to_dict(),
        "claim_by_age":       df.groupby("AGE")["OUTCOME"].mean().mul(100).round(1).to_dict(),
        "claim_by_exp":       df.groupby("DRIVING_EXPERIENCE")["OUTCOME"].mean().mul(100).round(1).to_dict(),
        "claim_by_income":    df.groupby("INCOME")["OUTCOME"].mean().mul(100).round(1).to_dict(),
        "model_name":         MODEL_NAME,
        "accuracy":           round(MODEL_ACC * 100, 2),
        "auc":                round(MODEL_AUC, 4),
        "features":           FEATURE_NAMES,
    }

STATS = _build_stats()

# ── Helper: encode one input row ──────────────────────────────────────────────
def _encode(payload: dict) -> np.ndarray:
    row = {}
    for feat in FEATURE_NAMES:
        val = payload.get(feat, "")
        if feat in ENCODERS:
            le = ENCODERS[feat]
            row[feat] = int(le.transform([val])[0]) if val in le.classes_ else 0
        else:
            try:
                row[feat] = float(val)
            except (ValueError, TypeError):
                row[feat] = 0.0
    return np.array([row[f] for f in FEATURE_NAMES]).reshape(1, -1)

# ── Routes ─────────────────────────────────────────────────────────────────────
@app.route("/predict", methods=["POST"])
def predict():
    """
    Accepts JSON body with feature key/value pairs.
    Returns { prediction, label, probability, risk_level }.
    """
    try:
        payload     = request.get_json(force=True)
        X           = _encode(payload)
        prediction  = int(MODEL.predict(X)[0])
        probability = float(MODEL.predict_proba(X)[0][1])
        risk        = ("High" if probability >= 0.65 else
                       "Medium" if probability >= 0.40 else "Low")
        return jsonify({
            "success":     True,
            "prediction":  prediction,
            "label":       "Claim" if prediction == 1 else "No Claim",
            "probability": round(probability * 100, 2),
            "risk_level":  risk,
        })
    except Exception as exc:
        return jsonify({"success": False, "error": str(exc)}), 400


@app.route("/stats", methods=["GET"])
def stats():
    """Returns dataset and model performance statistics."""
    return jsonify(STATS)


# ── Entry point ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"Model  : {MODEL_NAME}")
    print(f"Acc    : {MODEL_ACC:.4f}   AUC: {MODEL_AUC:.4f}")
    print("Flask API running on http://127.0.0.1:5000")
    app.run(debug=True, port=5000)


### Frontend - Streamlit - streamlit_app.py

In [ ]:
"""
streamlit_app.py  —  Streamlit Frontend
========================================
Two pages:
  1. Predict  — fill form, call Flask API, show result
  2. Dashboard — charts + model stats pulled from Flask API

Run (after Flask is running):
    streamlit run streamlit_app.py
"""

import os
import requests
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import streamlit as st

# ── Config ─────────────────────────────────────────────────────────────────────
API_BASE   = "http://127.0.0.1:5000"
BASE_DIR   = os.path.dirname(os.path.abspath(__file__))
EVAL_IMG   = os.path.join(BASE_DIR, "static", "model_evaluation.png")

st.set_page_config(
    page_title="Car Insurance Claim Predictor",
    page_icon="🚗",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Custom CSS ─────────────────────────────────────────────────────────────────
st.markdown("""
<style>
    .main-header {
        background: linear-gradient(135deg, #1e3a5f, #2d6a9f);
        color: white;
        padding: 1.4rem 2rem;
        border-radius: 10px;
        margin-bottom: 1.5rem;
    }
    .main-header h1 { margin: 0; font-size: 1.8rem; }
    .main-header p  { margin: .3rem 0 0; opacity: .85; font-size: .95rem; }

    .result-claim    { background:#fef2f2; border:2px solid #f87171;
                       border-radius:10px; padding:1.2rem 1.6rem; }
    .result-no-claim { background:#f0fdf4; border:2px solid #4ade80;
                       border-radius:10px; padding:1.2rem 1.6rem; }

    .kpi-box { background:#f7f8fa; border-radius:8px; padding:.9rem 1.2rem;
               border-left:4px solid #2d6a9f; }
    .kpi-val { font-size:1.8rem; font-weight:800; color:#1e3a5f; }
    .kpi-lbl { font-size:.75rem; color:#6b7280; text-transform:uppercase;
               letter-spacing:.5px; }

    div[data-testid="stSidebar"] { background:#1e3a5f; }
    div[data-testid="stSidebar"] * { color:#c9d8ea !important; }
    div[data-testid="stSidebar"] .css-1d391kg { background:#1e3a5f; }
</style>
""", unsafe_allow_html=True)

# ── Sidebar navigation ─────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown("### 🚗 Insurance Predictor")
    st.markdown("---")
    page = st.radio("Navigation", ["🔮 Predict", "📊 Dashboard"], label_visibility="collapsed")
    st.markdown("---")
    st.markdown("**Flask API**")
    try:
        r = requests.get(f"{API_BASE}/stats", timeout=3)
        if r.ok:
            _s = r.json()
            st.success("API Connected")
            st.caption(f"Model: {_s['model_name']}")
            st.caption(f"Accuracy: {_s['accuracy']}%")
        else:
            st.error("API error")
    except Exception:
        st.error("Flask API offline\nRun: `python app.py`")

# ══════════════════════════════════════════════════════════════════════════════
#  PAGE 1 — PREDICT
# ══════════════════════════════════════════════════════════════════════════════
if page == "🔮 Predict":

    st.markdown("""
    <div class="main-header">
      <h1>🔮 Car Insurance Claim Predictor</h1>
      <p>Fill in the driver and vehicle details below to predict whether a claim will be filed.</p>
    </div>
    """, unsafe_allow_html=True)

    with st.form("prediction_form"):
        st.subheader("👤 Personal Information")
        c1, c2, c3, c4 = st.columns(4)
        age        = c1.selectbox("Age Group",   ["16-25", "26-39", "40-64", "65+"])
        gender     = c2.selectbox("Gender",       ["male", "female"])
        race       = c3.selectbox("Race",         ["majority", "minority"])
        education  = c4.selectbox("Education",    ["none", "high school", "university"])

        c5, c6, c7, c8 = st.columns(4)
        income     = c5.selectbox("Income Class", ["poverty", "working class", "middle class", "upper class"])
        married    = c6.selectbox("Married",      [0, 1], format_func=lambda x: "Yes" if x else "No")
        children   = c7.selectbox("Has Children", [0, 1], format_func=lambda x: "Yes" if x else "No")
        credit     = c8.number_input("Credit Score (0–1)", min_value=0.0, max_value=1.0,
                                     value=0.55, step=0.01, format="%.3f")

        st.subheader("🚦 Driving Profile")
        d1, d2, d3, d4, d5 = st.columns(5)
        driving_exp   = d1.selectbox("Driving Experience", ["0-9y", "10-19y", "20-29y", "30y+"])
        speeding      = d2.number_input("Speeding Violations", min_value=0, value=0, step=1)
        duis          = d3.number_input("DUI Offences",        min_value=0, value=0, step=1)
        past_acc      = d4.number_input("Past Accidents",      min_value=0, value=0, step=1)
        annual_miles  = d5.number_input("Annual Mileage",      min_value=0, value=12000, step=500)

        st.subheader("🚗 Vehicle Details")
        v1, v2, v3, v4 = st.columns(4)
        vehicle_type  = v1.selectbox("Vehicle Type",       ["sedan", "sports car"])
        vehicle_year  = v2.selectbox("Vehicle Year",       ["before 2015", "after 2015"])
        vehicle_own   = v3.selectbox("Owns Vehicle",       [0, 1], format_func=lambda x: "Yes" if x else "No")
        postal        = v4.number_input("Postal Code",     min_value=0, value=10238, step=1)

        submitted = st.form_submit_button("🔍 Predict Claim", use_container_width=True,
                                          type="primary")

    if submitted:
        payload = {
            "AGE": age, "GENDER": gender, "RACE": race,
            "DRIVING_EXPERIENCE": driving_exp, "EDUCATION": education,
            "INCOME": income, "CREDIT_SCORE": credit,
            "VEHICLE_OWNERSHIP": float(vehicle_own),
            "VEHICLE_YEAR": vehicle_year, "MARRIED": float(married),
            "CHILDREN": float(children), "POSTAL_CODE": float(postal),
            "ANNUAL_MILEAGE": float(annual_miles),
            "VEHICLE_TYPE": vehicle_type,
            "SPEEDING_VIOLATIONS": float(speeding),
            "DUIS": float(duis), "PAST_ACCIDENTS": float(past_acc),
        }
        try:
            with st.spinner("Analysing..."):
                resp = requests.post(f"{API_BASE}/predict", json=payload, timeout=10)
            data = resp.json()
            if not data.get("success"):
                st.error(f"API Error: {data.get('error')}")
            else:
                is_claim = data["prediction"] == 1
                risk_colors = {"High": "🔴", "Medium": "🟡", "Low": "🟢"}
                risk_icon   = risk_colors.get(data["risk_level"], "⚪")

                box_class = "result-claim" if is_claim else "result-no-claim"
                icon      = "⚠️" if is_claim else "✅"
                st.markdown(f"""
                <div class="{box_class}">
                  <h2>{icon} Prediction: <strong>{data['label']}</strong></h2>
                  <p>The model predicts this driver <strong>{'WILL' if is_claim else 'WILL NOT'}</strong>
                     file an insurance claim.</p>
                </div>
                """, unsafe_allow_html=True)

                st.markdown("")
                m1, m2, m3 = st.columns(3)
                m1.metric("Claim Probability", f"{data['probability']}%")
                m2.metric("Risk Level",         f"{risk_icon} {data['risk_level']}")
                m3.metric("Predicted Outcome",  data["label"])

                st.progress(int(data["probability"]))

        except requests.exceptions.ConnectionError:
            st.error("Cannot connect to Flask API. Make sure `python app.py` is running on port 5000.")

# ══════════════════════════════════════════════════════════════════════════════
#  PAGE 2 — DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════
elif page == "📊 Dashboard":

    st.markdown("""
    <div class="main-header">
      <h1>📊 Analytics Dashboard</h1>
      <p>Dataset exploration and model performance overview.</p>
    </div>
    """, unsafe_allow_html=True)

    try:
        s = requests.get(f"{API_BASE}/stats", timeout=5).json()
    except Exception:
        st.error("Flask API offline. Run `python app.py` first.")
        st.stop()

    # ── KPI row ────────────────────────────────────────────────────────────────
    k1, k2, k3, k4, k5 = st.columns(5)
    k1.metric("Total Records",  f"{s['total']:,}")
    k2.metric("Claims Filed",   f"{s['claim_count']:,}", f"{s['claim_pct']}%")
    k3.metric("No Claims",      f"{s['no_claim_count']:,}")
    k4.metric("Model Accuracy", f"{s['accuracy']}%")
    k5.metric("ROC-AUC",        str(s['auc']))

    st.divider()

    # ── Model info ────────────────────────────────────────────────────────────
    st.subheader(f"🤖 Best Model: {s['model_name']}")
    if os.path.exists(EVAL_IMG):
        st.image(EVAL_IMG, caption="Confusion Matrix & Feature Importance", use_container_width=True)

    st.divider()

    # ── Charts row 1 ──────────────────────────────────────────────────────────
    st.subheader("📈 Dataset Distributions")
    col1, col2 = st.columns(2)

    with col1:
        st.markdown("**Claims vs No Claims**")
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.pie([s["no_claim_count"], s["claim_count"]],
               labels=["No Claim", "Claim"],
               colors=["#22c55e", "#ef4444"],
               autopct="%1.1f%%", startangle=90,
               wedgeprops=dict(edgecolor="white", linewidth=2))
        ax.set_title("Outcome Distribution", fontsize=11, fontweight="bold")
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with col2:
        st.markdown("**Age Group Distribution**")
        ages  = list(s["age_dist"].keys())
        acnts = list(s["age_dist"].values())
        fig, ax = plt.subplots(figsize=(5, 4))
        bars = ax.bar(ages, acnts, color="#2d6a9f", edgecolor="white")
        ax.bar_label(bars, padding=3, fontsize=9)
        ax.set_title("Customers by Age Group", fontsize=11, fontweight="bold")
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=15)
        st.pyplot(fig, use_container_width=True)
        plt.close()

    # ── Charts row 2 ──────────────────────────────────────────────────────────
    col3, col4 = st.columns(2)

    with col3:
        st.markdown("**Claim Rate by Age Group (%)**")
        ages_r = list(s["claim_by_age"].keys())
        rates  = list(s["claim_by_age"].values())
        fig, ax = plt.subplots(figsize=(5, 4))
        colors  = ["#ef4444" if r >= 40 else "#f59e0b" if r >= 25 else "#22c55e" for r in rates]
        bars    = ax.bar(ages_r, rates, color=colors, edgecolor="white")
        ax.bar_label(bars, labels=[f"{r}%" for r in rates], padding=3, fontsize=9)
        ax.set_title("Claim Rate by Age", fontsize=11, fontweight="bold")
        ax.set_ylabel("Claim Rate (%)")
        ax.set_ylim(0, max(rates) + 10)
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with col4:
        st.markdown("**Claim Rate by Driving Experience (%)**")
        exps   = list(s["claim_by_exp"].keys())
        erates = list(s["claim_by_exp"].values())
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.barh(exps, erates, color="#7c5cd8", edgecolor="white")
        for i, v in enumerate(erates):
            ax.text(v + 0.5, i, f"{v}%", va="center", fontsize=9)
        ax.set_title("Claim Rate by Driving Experience", fontsize=11, fontweight="bold")
        ax.set_xlabel("Claim Rate (%)")
        st.pyplot(fig, use_container_width=True)
        plt.close()

    # ── Charts row 3 ──────────────────────────────────────────────────────────
    col5, col6 = st.columns(2)

    with col5:
        st.markdown("**Claim Rate by Income Class (%)**")
        incs   = list(s["claim_by_income"].keys())
        irates = list(s["claim_by_income"].values())
        fig, ax = plt.subplots(figsize=(5, 4))
        palette = ["#1e3a5f", "#2d6a9f", "#06b6d4", "#7c5cd8"]
        bars    = ax.bar(incs, irates, color=palette[:len(incs)], edgecolor="white")
        ax.bar_label(bars, labels=[f"{r}%" for r in irates], padding=3, fontsize=9)
        ax.set_title("Claim Rate by Income", fontsize=11, fontweight="bold")
        ax.set_ylabel("Claim Rate (%)")
        ax.tick_params(axis="x", rotation=20)
        st.pyplot(fig, use_container_width=True)
        plt.close()

    with col6:
        st.markdown("**Income Distribution**")
        inc_keys = list(s["income_dist"].keys())
        inc_vals = list(s["income_dist"].values())
        fig, ax  = plt.subplots(figsize=(5, 4))
        ax.pie(inc_vals, labels=inc_keys,
               colors=["#1e3a5f","#2d6a9f","#06b6d4","#7c5cd8"],
               autopct="%1.1f%%", startangle=90,
               wedgeprops=dict(edgecolor="white", linewidth=2))
        ax.set_title("Income Class Distribution", fontsize=11, fontweight="bold")
        st.pyplot(fig, use_container_width=True)
        plt.close()

    # ── Raw dataset preview ────────────────────────────────────────────────────
    st.divider()
    st.subheader("📋 Dataset Preview")
    df_preview = pd.read_csv(os.path.join(BASE_DIR, "Car_Insurance_Claim.csv"))
    st.dataframe(df_preview.head(50), use_container_width=True)
    st.caption(f"Showing first 50 of {len(df_preview):,} rows   |   {df_preview.shape[1]} columns")


### Generate Screenshots for report - generate_screenshots.py

In [ ]:
"""
generate_screenshots.py
========================
Renders pixel-faithful mockup screenshots of every Streamlit UI page
using Matplotlib. No browser or running server needed.

Outputs (all saved to static/screenshots/):
  01_predict_form.png       — Predict page: empty form
  02_predict_result_claim.png    — Predict page: "Claim" result
  03_predict_result_no_claim.png — Predict page: "No Claim" result
  04_dashboard_kpi.png      — Dashboard page: KPI + model info
  05_dashboard_charts.png   — Dashboard page: EDA charts
  06_sidebar.png            — Sidebar navigation panel

Run:
    python generate_screenshots.py
"""

import os
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

BASE_DIR    = os.path.dirname(os.path.abspath(__file__))
SCREENS_DIR = os.path.join(BASE_DIR, "static", "screenshots")
os.makedirs(SCREENS_DIR, exist_ok=True)

# ── Load model & data ──────────────────────────────────────────────────────────
with open(os.path.join(BASE_DIR, "model.pkl"), "rb") as f:
    art = pickle.load(f)
MODEL         = art["model"]
ENCODERS      = art["encoders"]
FEATURE_NAMES = art["feature_names"]
MODEL_NAME    = art["model_name"]
MODEL_ACC     = art["accuracy"]
MODEL_AUC     = art["auc"]

df_raw = pd.read_csv(os.path.join(BASE_DIR, "Car_Insurance_Claim.csv"))
total  = len(df_raw)
claims = int(df_raw["OUTCOME"].sum())

# ── Shared drawing helpers ─────────────────────────────────────────────────────
NAV_BG      = "#1e3a5f"
PAGE_BG     = "#f0f4f8"
CARD_BG     = "#ffffff"
ACCENT      = "#2d6a9f"
GREEN       = "#22c55e"
RED         = "#ef4444"
PURPLE      = "#7c5cd8"
AMBER       = "#f59e0b"
TEXT        = "#1f2328"
MUTED       = "#6b7280"
BORDER      = "#e5e7eb"

def draw_browser_chrome(fig, ax_full, title="Car Insurance Claim Predictor — Streamlit"):
    """Draw a browser tab bar at the very top of the figure."""
    ax_full.set_xlim(0, 1); ax_full.set_ylim(0, 1)
    ax_full.axis("off")
    # browser bar
    ax_full.add_patch(FancyBboxPatch((0, 0.965), 1, 0.035,
        boxstyle="square,pad=0", fc="#e5e7eb", ec="none", zorder=5))
    # address bar
    ax_full.add_patch(FancyBboxPatch((0.18, 0.967), 0.64, 0.027,
        boxstyle="round,pad=0.004", fc="white", ec="#d1d5db", lw=0.8, zorder=6))
    ax_full.text(0.50, 0.9805, "localhost:8501", ha="center", va="center",
                 fontsize=7.5, color=MUTED, zorder=7)
    # three dots (close/min/max)
    for xi, col in [(0.025, "#ef4444"), (0.045, "#f59e0b"), (0.065, GREEN)]:
        circ = plt.Circle((xi, 0.9805), 0.008, color=col, zorder=7)
        ax_full.add_patch(circ)

def nav_sidebar(ax, active="predict"):
    """Draw the left sidebar."""
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1,
        boxstyle="square,pad=0", fc=NAV_BG, ec="none"))
    ax.text(0.5, 0.93, "🚗 Insurance", ha="center", va="center",
            fontsize=11, fontweight="bold", color="white")
    ax.text(0.5, 0.87, "Predictor", ha="center", va="center",
            fontsize=11, fontweight="bold", color="white")
    ax.axhline(0.83, color="#2d6a9f", lw=0.8, xmin=0.05, xmax=0.95)
    # nav items
    items = [("🔮 Predict", "predict", 0.75), ("📊 Dashboard", "dashboard", 0.67)]
    for label, key, y in items:
        is_active = (key == active)
        if is_active:
            ax.add_patch(FancyBboxPatch((0.05, y - 0.03), 0.90, 0.055,
                boxstyle="round,pad=0.01", fc="#2d6a9f", ec="none", alpha=0.6))
        ax.text(0.12, y + 0.005, label, ha="left", va="center",
                fontsize=9.5, color="white",
                fontweight="bold" if is_active else "normal")
    ax.axhline(0.60, color="#2d6a9f", lw=0.8, xmin=0.05, xmax=0.95)
    ax.text(0.5, 0.55, "Flask API", ha="center", va="center",
            fontsize=8, color="#c9d8ea", fontweight="bold")
    # API status badge
    ax.add_patch(FancyBboxPatch((0.1, 0.47), 0.80, 0.055,
        boxstyle="round,pad=0.01", fc="#166534", ec="none"))
    ax.text(0.5, 0.498, "✓  API Connected", ha="center", va="center",
            fontsize=8, color="white")
    ax.text(0.5, 0.43, f"Model: {MODEL_NAME}", ha="center", va="center",
            fontsize=7.5, color="#c9d8ea")
    ax.text(0.5, 0.39, f"Accuracy: {round(MODEL_ACC*100,1)}%", ha="center", va="center",
            fontsize=7.5, color="#c9d8ea")

def section_header(ax, x, y, w, h, text):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
        boxstyle="round,pad=0.008", fc=CARD_BG, ec=BORDER, lw=0.8))
    ax.text(x + 0.015, y + h/2, text, ha="left", va="center",
            fontsize=10, fontweight="bold", color=NAV_BG)

def form_field(ax, x, y, w, h, label, value, is_select=False):
    ax.text(x, y + h + 0.008, label.upper(), ha="left", va="bottom",
            fontsize=6.2, color=MUTED, fontweight="bold")
    ax.add_patch(FancyBboxPatch((x, y), w, h,
        boxstyle="round,pad=0.005", fc="#f9fafb", ec="#d1d5db", lw=0.8))
    ax.text(x + 0.01, y + h/2, value, ha="left", va="center",
            fontsize=7.5, color=TEXT)
    if is_select:
        ax.text(x + w - 0.012, y + h/2, "▾", ha="right", va="center",
                fontsize=7, color=MUTED)

def kpi_card(ax, x, y, w, h, value, label, color=ACCENT):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
        boxstyle="round,pad=0.01", fc="#f7f8fa", ec=BORDER, lw=0.8))
    ax.add_patch(plt.Rectangle((x, y), 0.006, h, fc=color, ec="none"))
    ax.text(x + w/2, y + h*0.62, str(value), ha="center", va="center",
            fontsize=13, fontweight="bold", color=NAV_BG)
    ax.text(x + w/2, y + h*0.22, label, ha="center", va="center",
            fontsize=6.5, color=MUTED, fontweight="bold")

# ══════════════════════════════════════════════════════════════════════════════
# SCREENSHOT 1 — Predict Page (form)
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering 01_predict_form.png ...")
fig = plt.figure(figsize=(14, 9), facecolor=PAGE_BG)

# Layout: sidebar (18%) | main (82%)
ax_browser = fig.add_axes([0, 0, 1, 1], zorder=0)
ax_browser.set_facecolor(PAGE_BG); ax_browser.axis("off")
draw_browser_chrome(fig, ax_browser)

ax_side = fig.add_axes([0, 0, 0.18, 0.965])
nav_sidebar(ax_side, active="predict")

ax = fig.add_axes([0.19, 0.02, 0.80, 0.935])
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_facecolor(PAGE_BG)

# Hero banner
ax.add_patch(FancyBboxPatch((0, 0.90), 1.0, 0.098,
    boxstyle="round,pad=0.01", fc=NAV_BG, ec="none"))
ax.text(0.5, 0.955, "🔮  Car Insurance Claim Predictor", ha="center", va="center",
        fontsize=14, fontweight="bold", color="white")
ax.text(0.5, 0.920, "Fill in driver and vehicle details below to predict whether a claim will be filed.",
        ha="center", va="center", fontsize=8.5, color="#c9d8ea")

# Main form card
ax.add_patch(FancyBboxPatch((0.01, 0.01), 0.98, 0.875,
    boxstyle="round,pad=0.01", fc=CARD_BG, ec=BORDER, lw=0.8))

# Section: Personal Info
section_header(ax, 0.03, 0.775, 0.94, 0.028, "👤  Personal Information")

field_data_row1 = [
    ("Age Group",    "16-25",         True),
    ("Gender",       "male",          True),
    ("Race",         "majority",      True),
    ("Education",    "high school",   True),
]
xs = [0.04, 0.27, 0.50, 0.73]
for (lbl, val, sel), x in zip(field_data_row1, xs):
    form_field(ax, x, 0.715, 0.21, 0.040, lbl, val, is_select=sel)

field_data_row2 = [
    ("Income Class",  "poverty",   True),
    ("Married",       "No",        True),
    ("Has Children",  "No",        True),
    ("Credit Score",  "0.350",     False),
]
for (lbl, val, sel), x in zip(field_data_row2, xs):
    form_field(ax, x, 0.645, 0.21, 0.040, lbl, val, is_select=sel)

# Section: Driving Profile
section_header(ax, 0.03, 0.600, 0.94, 0.028, "🚦  Driving Profile")

drive_fields = [
    ("Driving Experience", "0-9y",  True),
    ("Speeding Violations","2",     False),
    ("DUI Offences",       "0",     False),
    ("Past Accidents",     "1",     False),
    ("Annual Mileage",     "16000", False),
]
xs5 = [0.04, 0.23, 0.42, 0.61, 0.80]
for (lbl, val, sel), x in zip(drive_fields, xs5):
    form_field(ax, x, 0.540, 0.17, 0.040, lbl, val, is_select=sel)

# Section: Vehicle Details
section_header(ax, 0.03, 0.496, 0.94, 0.028, "🚗  Vehicle Details")

veh_fields = [
    ("Vehicle Type",  "sedan",        True),
    ("Vehicle Year",  "before 2015",  True),
    ("Owns Vehicle",  "No",           True),
    ("Postal Code",   "10238",        False),
]
for (lbl, val, sel), x in zip(veh_fields, xs):
    form_field(ax, x, 0.436, 0.21, 0.040, lbl, val, is_select=sel)

# Predict button
ax.add_patch(FancyBboxPatch((0.03, 0.380), 0.94, 0.042,
    boxstyle="round,pad=0.008", fc=NAV_BG, ec="none"))
ax.text(0.5, 0.402, "🔍  Predict Claim", ha="center", va="center",
        fontsize=11, fontweight="bold", color="white")

plt.savefig(os.path.join(SCREENS_DIR, "01_predict_form.png"),
            dpi=130, bbox_inches="tight", facecolor=PAGE_BG)
plt.close()
print("  saved.")

# ══════════════════════════════════════════════════════════════════════════════
# SCREENSHOT 2 — Predict Page: CLAIM result
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering 02_predict_result_claim.png ...")
fig = plt.figure(figsize=(14, 9), facecolor=PAGE_BG)
ax_browser = fig.add_axes([0, 0, 1, 1], zorder=0)
ax_browser.set_facecolor(PAGE_BG); ax_browser.axis("off")
draw_browser_chrome(fig, ax_browser)
ax_side = fig.add_axes([0, 0, 0.18, 0.965])
nav_sidebar(ax_side, active="predict")
ax = fig.add_axes([0.19, 0.02, 0.80, 0.935])
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_facecolor(PAGE_BG)

# Hero
ax.add_patch(FancyBboxPatch((0, 0.90), 1.0, 0.098,
    boxstyle="round,pad=0.01", fc=NAV_BG, ec="none"))
ax.text(0.5, 0.955, "🔮  Car Insurance Claim Predictor", ha="center", va="center",
        fontsize=14, fontweight="bold", color="white")
ax.text(0.5, 0.920, "Fill in driver and vehicle details below to predict whether a claim will be filed.",
        ha="center", va="center", fontsize=8.5, color="#c9d8ea")

# Compact form (greyed out / submitted state)
ax.add_patch(FancyBboxPatch((0.01, 0.46), 0.98, 0.42,
    boxstyle="round,pad=0.01", fc=CARD_BG, ec=BORDER, lw=0.8, alpha=0.6))
ax.text(0.5, 0.78, "[ Form submitted — see prediction below ]",
        ha="center", va="center", fontsize=8, color=MUTED, style="italic")

# ── RESULT BOX: Claim ──
ax.add_patch(FancyBboxPatch((0.01, 0.255), 0.98, 0.185,
    boxstyle="round,pad=0.012", fc="#fef2f2", ec="#f87171", lw=2.0))
ax.text(0.06, 0.420, "⚠️", fontsize=22, va="center")
ax.text(0.16, 0.425, "Prediction:", fontsize=10, color=MUTED, va="center")
ax.text(0.16, 0.395, "CLAIM", fontsize=19, fontweight="bold", color="#dc2626", va="center")
ax.text(0.16, 0.372, "The model predicts this driver WILL file an insurance claim.",
        fontsize=8.5, color=TEXT, va="center")

# Three metric columns
for x, val, lbl in [(0.03, "78.4%", "Claim Probability"),
                    (0.37, "🔴 High", "Risk Level"),
                    (0.71, "Claim", "Predicted Outcome")]:
    ax.add_patch(FancyBboxPatch((x, 0.263), 0.295, 0.075,
        boxstyle="round,pad=0.008", fc="white", ec=BORDER, lw=0.8))
    ax.text(x + 0.148, 0.305, val, ha="center", va="center",
            fontsize=13, fontweight="bold", color="#dc2626" if "%" in val or "Claim" in val else TEXT)
    ax.text(x + 0.148, 0.275, lbl, ha="center", va="center",
            fontsize=7, color=MUTED, fontweight="bold")

# Progress bar
ax.add_patch(FancyBboxPatch((0.01, 0.248), 0.98, 0.012,
    boxstyle="square,pad=0", fc=BORDER, ec="none"))
ax.add_patch(FancyBboxPatch((0.01, 0.248), 0.98*0.784, 0.012,
    boxstyle="square,pad=0", fc="#ef4444", ec="none"))

plt.savefig(os.path.join(SCREENS_DIR, "02_predict_result_claim.png"),
            dpi=130, bbox_inches="tight", facecolor=PAGE_BG)
plt.close()
print("  saved.")

# ══════════════════════════════════════════════════════════════════════════════
# SCREENSHOT 3 — Predict Page: NO CLAIM result
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering 03_predict_result_no_claim.png ...")
fig = plt.figure(figsize=(14, 9), facecolor=PAGE_BG)
ax_browser = fig.add_axes([0, 0, 1, 1], zorder=0)
ax_browser.set_facecolor(PAGE_BG); ax_browser.axis("off")
draw_browser_chrome(fig, ax_browser)
ax_side = fig.add_axes([0, 0, 0.18, 0.965])
nav_sidebar(ax_side, active="predict")
ax = fig.add_axes([0.19, 0.02, 0.80, 0.935])
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_facecolor(PAGE_BG)

ax.add_patch(FancyBboxPatch((0, 0.90), 1.0, 0.098,
    boxstyle="round,pad=0.01", fc=NAV_BG, ec="none"))
ax.text(0.5, 0.955, "🔮  Car Insurance Claim Predictor", ha="center", va="center",
        fontsize=14, fontweight="bold", color="white")
ax.text(0.5, 0.920, "Fill in driver and vehicle details below to predict whether a claim will be filed.",
        ha="center", va="center", fontsize=8.5, color="#c9d8ea")

ax.add_patch(FancyBboxPatch((0.01, 0.46), 0.98, 0.42,
    boxstyle="round,pad=0.01", fc=CARD_BG, ec=BORDER, lw=0.8, alpha=0.6))
ax.text(0.5, 0.78, "[ Form submitted — see prediction below ]",
        ha="center", va="center", fontsize=8, color=MUTED, style="italic")

# ── RESULT BOX: No Claim ──
ax.add_patch(FancyBboxPatch((0.01, 0.255), 0.98, 0.185,
    boxstyle="round,pad=0.012", fc="#f0fdf4", ec="#4ade80", lw=2.0))
ax.text(0.06, 0.420, "✅", fontsize=22, va="center")
ax.text(0.16, 0.425, "Prediction:", fontsize=10, color=MUTED, va="center")
ax.text(0.16, 0.395, "NO CLAIM", fontsize=19, fontweight="bold", color="#16a34a", va="center")
ax.text(0.16, 0.372, "The model predicts this driver WILL NOT file an insurance claim.",
        fontsize=8.5, color=TEXT, va="center")

for x, val, lbl, col in [(0.03, "18.2%", "Claim Probability", "#16a34a"),
                          (0.37, "🟢 Low", "Risk Level", TEXT),
                          (0.71, "No Claim", "Predicted Outcome", "#16a34a")]:
    ax.add_patch(FancyBboxPatch((x, 0.263), 0.295, 0.075,
        boxstyle="round,pad=0.008", fc="white", ec=BORDER, lw=0.8))
    ax.text(x + 0.148, 0.305, val, ha="center", va="center",
            fontsize=13, fontweight="bold", color=col)
    ax.text(x + 0.148, 0.275, lbl, ha="center", va="center",
            fontsize=7, color=MUTED, fontweight="bold")

ax.add_patch(FancyBboxPatch((0.01, 0.248), 0.98, 0.012,
    boxstyle="square,pad=0", fc=BORDER, ec="none"))
ax.add_patch(FancyBboxPatch((0.01, 0.248), 0.98*0.182, 0.012,
    boxstyle="square,pad=0", fc=GREEN, ec="none"))

plt.savefig(os.path.join(SCREENS_DIR, "03_predict_result_no_claim.png"),
            dpi=130, bbox_inches="tight", facecolor=PAGE_BG)
plt.close()
print("  saved.")

# ══════════════════════════════════════════════════════════════════════════════
# SCREENSHOT 4 — Dashboard: KPI + model card
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering 04_dashboard_kpi.png ...")
fig = plt.figure(figsize=(14, 9), facecolor=PAGE_BG)
ax_browser = fig.add_axes([0, 0, 1, 1], zorder=0)
ax_browser.set_facecolor(PAGE_BG); ax_browser.axis("off")
draw_browser_chrome(fig, ax_browser)
ax_side = fig.add_axes([0, 0, 0.18, 0.965])
nav_sidebar(ax_side, active="dashboard")
ax = fig.add_axes([0.19, 0.02, 0.80, 0.935])
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
ax.set_facecolor(PAGE_BG)

# Hero
ax.add_patch(FancyBboxPatch((0, 0.90), 1.0, 0.098,
    boxstyle="round,pad=0.01", fc=NAV_BG, ec="none"))
ax.text(0.5, 0.955, "📊  Analytics Dashboard", ha="center", va="center",
        fontsize=14, fontweight="bold", color="white")
ax.text(0.5, 0.920, "Dataset exploration and model performance overview.",
        ha="center", va="center", fontsize=8.5, color="#c9d8ea")

# KPI row
kpi_items = [
    (f"{total:,}",                      "Total Records",    ACCENT),
    (f"{claims:,}",                     "Claims Filed",     RED),
    (f"{total-claims:,}",               "No Claims",        GREEN),
    (f"{round(MODEL_ACC*100,2)}%",      "Model Accuracy",   PURPLE),
    (f"{round(MODEL_AUC,4)}",           "ROC-AUC",          AMBER),
]
kpi_w = 0.178
for i, (val, lbl, col) in enumerate(kpi_items):
    kpi_card(ax, 0.02 + i*(kpi_w + 0.015), 0.805, kpi_w, 0.075, val, lbl, col)

# Model card
ax.add_patch(FancyBboxPatch((0.02, 0.545), 0.96, 0.240,
    boxstyle="round,pad=0.01", fc=CARD_BG, ec=BORDER, lw=0.8))
ax.text(0.035, 0.768, f"🤖  Best Model:  {MODEL_NAME}", ha="left", va="center",
        fontsize=11, fontweight="bold", color=NAV_BG)
ax.axhline(0.755, color=BORDER, lw=0.8, xmin=0.03, xmax=0.97)

# Mini metric pills inside model card
pills = [
    (f"{round(MODEL_ACC*100,2)}%", "Accuracy"),
    (f"{round(MODEL_AUC,4)}",      "ROC-AUC"),
    (f"{total:,}",                 "Training Records"),
    ("18",                         "Features Used"),
]
pw = 0.19
for i, (v, l) in enumerate(pills):
    px = 0.04 + i * (pw + 0.03)
    ax.add_patch(FancyBboxPatch((px, 0.590), pw, 0.130,
        boxstyle="round,pad=0.01", fc="#f0f4f8", ec=BORDER, lw=0.6))
    ax.text(px + pw/2, 0.670, v,  ha="center", va="center",
            fontsize=15, fontweight="bold", color=NAV_BG)
    ax.text(px + pw/2, 0.612, l, ha="center", va="center",
            fontsize=7, color=MUTED, fontweight="bold")

# eval image thumbnail
eval_img_path = os.path.join(BASE_DIR, "static", "model_evaluation.png")
eval_img = plt.imread(eval_img_path)
ax_eval = fig.add_axes([0.19 + 0.80*0.02, 0.02 + 0.935*0.05, 0.80*0.96, 0.935*0.47])
ax_eval.imshow(eval_img, aspect="auto")
ax_eval.set_title("Confusion Matrix & Feature Importance (from training)",
                   fontsize=8, color=MUTED, pad=4)
ax_eval.axis("off")

plt.savefig(os.path.join(SCREENS_DIR, "04_dashboard_kpi.png"),
            dpi=130, bbox_inches="tight", facecolor=PAGE_BG)
plt.close()
print("  saved.")

# ══════════════════════════════════════════════════════════════════════════════
# SCREENSHOT 5 — Dashboard: EDA Charts grid
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering 05_dashboard_charts.png ...")

fig = plt.figure(figsize=(14, 10), facecolor=PAGE_BG)
fig.suptitle("📊  Analytics Dashboard  —  Dataset Distributions",
             fontsize=13, fontweight="bold", color=NAV_BG, y=0.98)

axes_pos = [
    [0.04,  0.54, 0.43, 0.40],  # top-left
    [0.53,  0.54, 0.43, 0.40],  # top-right
    [0.04,  0.06, 0.43, 0.40],  # bottom-left
    [0.53,  0.06, 0.43, 0.40],  # bottom-right
]

# Chart A — Outcome pie
ax0 = fig.add_axes(axes_pos[0])
ax0.set_facecolor(CARD_BG)
wedges, texts, auto = ax0.pie(
    [total - claims, claims],
    labels=["No Claim", "Claim"],
    colors=[GREEN, RED],
    autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    textprops=dict(fontsize=9))
ax0.set_title("Claims vs No Claims", fontsize=11, fontweight="bold", color=NAV_BG, pad=8)

# Chart B — Claim rate by age
ax1 = fig.add_axes(axes_pos[1])
ax1.set_facecolor(CARD_BG)
cr_age = df_raw.groupby("AGE")["OUTCOME"].mean().mul(100).round(1)
bars = ax1.bar(cr_age.index, cr_age.values, color=ACCENT, edgecolor="white", width=0.6)
ax1.bar_label(bars, labels=[f"{v}%" for v in cr_age.values], padding=3, fontsize=8.5)
ax1.set_title("Claim Rate by Age Group", fontsize=11, fontweight="bold", color=NAV_BG, pad=8)
ax1.set_ylabel("Claim Rate (%)", fontsize=8.5)
ax1.set_ylim(0, max(cr_age.values) + 12)
ax1.tick_params(axis="both", labelsize=8)
ax1.set_facecolor(CARD_BG)
ax1.spines[["top","right"]].set_visible(False)

# Chart C — Claim rate by driving experience
ax2 = fig.add_axes(axes_pos[2])
ax2.set_facecolor(CARD_BG)
cr_exp = df_raw.groupby("DRIVING_EXPERIENCE")["OUTCOME"].mean().mul(100).round(1)
ax2.barh(cr_exp.index, cr_exp.values, color=PURPLE, edgecolor="white", height=0.6)
for i, v in enumerate(cr_exp.values):
    ax2.text(v + 0.8, i, f"{v}%", va="center", fontsize=8.5)
ax2.set_title("Claim Rate by Driving Experience", fontsize=11, fontweight="bold", color=NAV_BG, pad=8)
ax2.set_xlabel("Claim Rate (%)", fontsize=8.5)
ax2.tick_params(axis="both", labelsize=8)
ax2.spines[["top","right"]].set_visible(False)

# Chart D — Income distribution
ax3 = fig.add_axes(axes_pos[3])
ax3.set_facecolor(CARD_BG)
inc = df_raw["INCOME"].value_counts()
inc_colors = [NAV_BG, ACCENT, "#06b6d4", PURPLE]
ax3.pie(inc.values, labels=inc.index,
        colors=inc_colors[:len(inc)],
        autopct="%1.1f%%", startangle=90,
        wedgeprops=dict(edgecolor="white", linewidth=2),
        textprops=dict(fontsize=8.5))
ax3.set_title("Income Class Distribution", fontsize=11, fontweight="bold", color=NAV_BG, pad=8)

plt.savefig(os.path.join(SCREENS_DIR, "05_dashboard_charts.png"),
            dpi=130, bbox_inches="tight", facecolor=PAGE_BG)
plt.close()
print("  saved.")

# ══════════════════════════════════════════════════════════════════════════════
# SCREENSHOT 6 — Dataset preview table
# ══════════════════════════════════════════════════════════════════════════════
print("Rendering 06_dataset_preview.png ...")
fig, ax = plt.subplots(figsize=(14, 5), facecolor=CARD_BG)
ax.set_facecolor(CARD_BG)
ax.axis("off")
ax.set_title("📋  Dataset Preview  —  Car_Insurance_Claim.csv  (first 8 rows)",
             fontsize=11, fontweight="bold", color=NAV_BG, pad=10, loc="left")

preview = df_raw.head(8)
cols_show = ["ID","AGE","GENDER","DRIVING_EXPERIENCE","INCOME",
             "CREDIT_SCORE","SPEEDING_VIOLATIONS","PAST_ACCIDENTS","OUTCOME"]
preview = preview[cols_show].copy()
preview["CREDIT_SCORE"] = preview["CREDIT_SCORE"].round(3)

col_widths = [0.09, 0.07, 0.07, 0.12, 0.11, 0.10, 0.13, 0.11, 0.08]
xs = [sum(col_widths[:i]) + 0.02 for i in range(len(col_widths))]

# Header row
for j, (col, x) in enumerate(zip(cols_show, xs)):
    ax.add_patch(FancyBboxPatch((x, 0.82), col_widths[j]-0.005, 0.10,
        boxstyle="square,pad=0", fc=NAV_BG, ec="white", lw=0.5,
        transform=ax.transAxes))
    ax.text(x + (col_widths[j]-0.005)/2, 0.87, col.replace("_","\n"),
            ha="center", va="center", fontsize=6.5, fontweight="bold",
            color="white", transform=ax.transAxes)

# Data rows
row_colors = [CARD_BG, "#f7f8fa"]
for i, (_, row) in enumerate(preview.iterrows()):
    yb = 0.82 - (i+1)*0.095
    for j, (col, x) in enumerate(zip(cols_show, xs)):
        ax.add_patch(FancyBboxPatch((x, yb), col_widths[j]-0.005, 0.090,
            boxstyle="square,pad=0", fc=row_colors[i % 2], ec=BORDER, lw=0.3,
            transform=ax.transAxes))
        val = str(row[col])
        if col == "OUTCOME":
            fc = "#fee2e2" if val == "1.0" else "#dcfce7"
            ax.add_patch(FancyBboxPatch((x+0.005, yb+0.015),
                col_widths[j]-0.015, 0.058,
                boxstyle="round,pad=0.005", fc=fc, ec="none",
                transform=ax.transAxes))
        ax.text(x + (col_widths[j]-0.005)/2, yb + 0.042, val,
                ha="center", va="center", fontsize=6.8, color=TEXT,
                transform=ax.transAxes)

plt.tight_layout(pad=0.5)
plt.savefig(os.path.join(SCREENS_DIR, "06_dataset_preview.png"),
            dpi=130, bbox_inches="tight", facecolor=CARD_BG)
plt.close()
print("  saved.")

print(f"\nAll screenshots saved to: {SCREENS_DIR}")
print("Files:")
for f in sorted(os.listdir(SCREENS_DIR)):
    path = os.path.join(SCREENS_DIR, f)
    print(f"  {f}  ({os.path.getsize(path)//1024} KB)")


### Generate Report - Docx - generate_docx.py

In [ ]:
"""
generate_docx.py
================
Generates a comprehensive .docx project report with all sections:
Dataset Info, Dataset Insights, Model Info, Model Insights,
Business Insights, Business Analysis, UI Screenshots, Future Scope.

Run:
    python generate_docx.py
    # outputs: project_report.docx
"""

import os, io, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc as sk_auc)
from sklearn.model_selection import train_test_split
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

BASE_DIR    = os.path.dirname(os.path.abspath(__file__))
SCREENS_DIR = os.path.join(BASE_DIR, "static", "screenshots")

# ── Load artefacts ─────────────────────────────────────────────────────────────
with open(os.path.join(BASE_DIR, "model.pkl"), "rb") as f:
    art = pickle.load(f)
MODEL, ENCODERS = art["model"], art["encoders"]
FEATURE_NAMES   = art["feature_names"]
MODEL_NAME      = art["model_name"]
MODEL_ACC       = art["accuracy"]
MODEL_AUC       = art["auc"]

# ── Load & prep data ────────────────────────────────────────────────────────────
df_raw = pd.read_csv(os.path.join(BASE_DIR, "Car_Insurance_Claim.csv"))
df = df_raw.copy()
df.drop(columns=["ID"], inplace=True)
for col in df.columns:
    if df[col].dtype == "object":
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

CAT_COLS = ["AGE","GENDER","RACE","DRIVING_EXPERIENCE","EDUCATION","INCOME","VEHICLE_YEAR","VEHICLE_TYPE"]
for col in CAT_COLS:
    df[col] = ENCODERS[col].transform(df[col].astype(str))

X = df.drop(columns=["OUTCOME"])
y = df["OUTCOME"].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_pred  = MODEL.predict(X_te)
y_prob  = MODEL.predict_proba(X_te)[:, 1]
cr_dict = classification_report(y_te, y_pred, target_names=["No Claim","Claim"], output_dict=True)
cm_vals = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm_vals.ravel()
specificity = round(tn / (tn + fp) * 100, 1)
fpr, tpr, _  = roc_curve(y_te, y_prob)
roc_auc_val  = sk_auc(fpr, tpr)
fi_series    = pd.Series(MODEL.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)

total  = len(df_raw)
claims = int(df_raw["OUTCOME"].sum())
no_cl  = total - claims

cr_age = df_raw.groupby("AGE")["OUTCOME"].mean().mul(100).round(1)
cr_exp = df_raw.groupby("DRIVING_EXPERIENCE")["OUTCOME"].mean().mul(100).round(1)
cr_inc = df_raw.groupby("INCOME")["OUTCOME"].mean().mul(100).round(1).sort_values(ascending=False)
cr_gen = df_raw.groupby("GENDER")["OUTCOME"].mean().mul(100).round(1)
cr_veh = df_raw.groupby("VEHICLE_TYPE")["OUTCOME"].mean().mul(100).round(1)
cr_yr  = df_raw.groupby("VEHICLE_YEAR")["OUTCOME"].mean().mul(100).round(1)

numeric_cols = ["CREDIT_SCORE","ANNUAL_MILEAGE","SPEEDING_VIOLATIONS","DUIS","PAST_ACCIDENTS"]
desc = df_raw[numeric_cols].describe().round(3)

# ── Helpers ────────────────────────────────────────────────────────────────────
def fig_buf(fig, dpi=110):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    buf.seek(0)
    return buf

def screen_buf(filename):
    path = os.path.join(SCREENS_DIR, filename)
    with open(path, "rb") as f:
        return io.BytesIO(f.read())

def shade_row(row, hex_color="1E3A5F"):
    for cell in row.cells:
        tc   = cell._tc
        tcPr = tc.get_or_add_tcPr()
        shd  = OxmlElement("w:shd")
        shd.set(qn("w:val"),   "clear")
        shd.set(qn("w:color"), "auto")
        shd.set(qn("w:fill"),  hex_color)
        tcPr.append(shd)
        for para in cell.paragraphs:
            for run in para.runs:
                run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
                run.font.bold = True

def add_heading(doc, text, level=1):
    h = doc.add_heading(text, level=level)
    h.alignment = WD_ALIGN_PARAGRAPH.LEFT
    for run in h.runs:
        run.font.color.rgb = RGBColor(0x1E, 0x3A, 0x5F)
    return h

def add_para(doc, text, size=11):
    p = doc.add_paragraph(text)
    for run in p.runs:
        run.font.size = Pt(size)
    return p

def add_caption(doc, text):
    p = doc.add_paragraph(text)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.runs[0]; r.font.size = Pt(9)
    r.font.color.rgb = RGBColor(0x6B, 0x72, 0x80); r.font.italic = True
    return p

def add_screenshot(doc, filename, caption_text, width=Inches(5.8)):
    doc.add_picture(screen_buf(filename), width=width)
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    add_caption(doc, caption_text)
    doc.add_paragraph()

def add_table(doc, rows_data, col_widths=None):
    t = doc.add_table(rows=len(rows_data), cols=len(rows_data[0]))
    t.style = "Table Grid"
    for i, row_data in enumerate(rows_data):
        for j, val in enumerate(row_data):
            t.rows[i].cells[j].text = str(val)
        if i == 0:
            shade_row(t.rows[i])
    return t

def add_bullet(doc, items):
    for item in items:
        p = doc.add_paragraph(item, style="List Bullet")
        p.runs[0].font.size = Pt(10.5)

def add_callout(doc, text, prefix="NOTE"):
    p = doc.add_paragraph()
    run = p.add_run(f"{prefix}: {text}")
    run.font.size = Pt(10); run.font.italic = True
    run.font.color.rgb = RGBColor(0x1E, 0x40, 0xAF)

# ──────────────────────────────────────────────────────────────────────────────
# BUILD DOCUMENT
# ──────────────────────────────────────────────────────────────────────────────
doc = Document()
for section in doc.sections:
    section.top_margin    = Inches(1.0)
    section.bottom_margin = Inches(1.0)
    section.left_margin   = Inches(1.1)
    section.right_margin  = Inches(1.1)

# ── COVER ──────────────────────────────────────────────────────────────────────
doc.add_paragraph()
title = doc.add_heading("Car Insurance Claim Prediction", level=0)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER
for run in title.runs:
    run.font.color.rgb = RGBColor(0x1E, 0x3A, 0x5F); run.font.size = Pt(26)

for txt, sz, italic in [
    ("A Comprehensive End-to-End Machine Learning Project", 14, False),
    ("Flask REST API Backend  +  Streamlit Frontend UI",     12, True),
]:
    p = doc.add_paragraph(txt); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.runs[0]; r.font.size = Pt(sz)
    r.font.color.rgb = RGBColor(0x6B, 0x72, 0x80); r.font.italic = italic

doc.add_paragraph()
for line in [
    f"Best Model  :  {MODEL_NAME}",
    f"Accuracy    :  {round(MODEL_ACC*100,2)}%     |     ROC-AUC: {round(MODEL_AUC,4)}",
    f"Dataset     :  {total:,} records  |  18 features  |  {round(claims/total*100,1)}% claim rate",
    "Course      :  IBM AICTE AI/ML Internship",
]:
    p = doc.add_paragraph(line); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.runs[0].font.size = Pt(11)

doc.add_page_break()

# ── 1. PROJECT OVERVIEW ────────────────────────────────────────────────────────
add_heading(doc, "1. Project Overview")
add_para(doc,
    "Car insurance companies face significant financial risk from high-frequency or fraudulent "
    "claims. Predicting whether a policyholder will file a claim allows insurers to price risk "
    "accurately, prioritise underwriting reviews, and intervene proactively with high-risk customers.")
add_para(doc,
    "This project builds a production-ready binary classification pipeline that takes "
    "17 policyholder attributes and outputs a claim probability, a prediction label (Claim / No Claim), "
    "and a risk tier (Low / Medium / High).")

add_heading(doc, "System Components", level=2)
add_bullet(doc, [
    "train_model.py — data ingestion, preprocessing, multi-model training, auto-selection, model.pkl export",
    "app.py — Flask REST API: POST /predict and GET /stats endpoints (port 5000)",
    "streamlit_app.py — Streamlit web UI: Predict page + Analytics Dashboard (port 8501)",
    "generate_screenshots.py — renders 6 UI mockup PNGs using Matplotlib",
    "generate_report.py — self-contained HTML report (all charts base64-embedded)",
    "generate_docx.py — this Word document report",
])

add_heading(doc, "Technology Stack", level=2)
add_table(doc, [
    ("Layer",           "Technology",       "Purpose"),
    ("ML Modelling",    "scikit-learn",     "Random Forest, Gradient Boosting, Logistic Regression + 5-fold CV"),
    ("Backend API",     "Flask + flask-cors","REST API, JSON in/out, CORS-enabled, port 5000"),
    ("Frontend UI",     "Streamlit",        "Interactive web app: Predict + Dashboard pages, port 8501"),
    ("Data Processing", "pandas, NumPy",    "ETL, feature engineering, null imputation"),
    ("Visualisation",   "Matplotlib, Seaborn","EDA charts, model evaluation plots, UI mockups"),
    ("Reports",         "Python",           "HTML report (generate_report.py), Word report (generate_docx.py)"),
])

# ── 2. DATASET INFORMATION ─────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "2. Dataset Information")

add_heading(doc, "2.1 — Dataset Summary", level=2)
add_table(doc, [
    ("Metric",         "Value"),
    ("Source File",    "Car_Insurance_Claim.csv"),
    ("Total Records",  f"{total:,}"),
    ("Input Features", "17  (8 categorical, 5 numeric, 4 binary)"),
    ("Target Column",  "OUTCOME  (0 = no claim, 1 = claim filed)"),
    ("Claims (=1)",    f"{claims:,}  ({round(claims/total*100,1)}%)"),
    ("No Claims (=0)", f"{no_cl:,}  ({round(no_cl/total*100,1)}%)"),
    ("Missing Values", f"{int(df_raw.isnull().sum().sum())} total (CREDIT_SCORE, ANNUAL_MILEAGE)"),
    ("Imputation",     "Median for numeric, mode for categorical"),
    ("Train Split",    "8,000 records (80%, stratified)"),
    ("Test Split",     "2,000 records (20%, stratified)"),
])

add_heading(doc, "2.2 — Dataset Preview", level=2)
add_para(doc,
    "The screenshot below shows the first 8 rows of Car_Insurance_Claim.csv as rendered inside "
    "the Streamlit Dashboard page. The OUTCOME column is colour-coded: "
    "red background = claim filed (1.0), green = no claim (0.0).")
add_screenshot(doc, "06_dataset_preview.png",
    "Fig 1 — First 8 rows of Car_Insurance_Claim.csv (as shown in Streamlit Dashboard)")

add_heading(doc, "2.3 — Feature Dictionary", level=2)
add_table(doc, [
    ("Feature",            "Type",        "Values",                        "Description"),
    ("AGE",                "Categorical", "16-25, 26-39, 40-64, 65+",     "Driver age group"),
    ("GENDER",             "Categorical", "male, female",                  "Driver gender"),
    ("RACE",               "Categorical", "majority, minority",            "Race category"),
    ("DRIVING_EXPERIENCE", "Categorical", "0-9y, 10-19y, 20-29y, 30y+",  "Years of driving experience"),
    ("EDUCATION",          "Categorical", "none, high school, university", "Highest education level"),
    ("INCOME",             "Categorical", "poverty → upper class",         "Income bracket"),
    ("VEHICLE_YEAR",       "Categorical", "before / after 2015",           "Vehicle manufacture year band"),
    ("VEHICLE_TYPE",       "Categorical", "sedan, sports car",             "Type of vehicle insured"),
    ("CREDIT_SCORE",       "Numeric",     "0.0 – 1.0",                    "Normalised credit score"),
    ("ANNUAL_MILEAGE",     "Numeric",     "~5,000 – 25,000 km",           "Kilometres driven per year"),
    ("SPEEDING_VIOLATIONS","Numeric",     "0 – 20+",                      "Number of speeding tickets"),
    ("DUIS",               "Numeric",     "0 – 5",                        "DUI offences recorded"),
    ("PAST_ACCIDENTS",     "Numeric",     "0 – 15",                       "Prior accident count"),
    ("VEHICLE_OWNERSHIP",  "Binary",      "0 / 1",                        "Whether driver owns vehicle"),
    ("MARRIED",            "Binary",      "0 / 1",                        "Marital status"),
    ("CHILDREN",           "Binary",      "0 / 1",                        "Has dependent children"),
    ("POSTAL_CODE",        "Numeric",     "Discrete codes",               "Area postal code"),
    ("OUTCOME",            "TARGET",      "0 / 1",                        "1 = claim filed, 0 = no claim"),
])

add_heading(doc, "2.4 — Numeric Feature Statistics", level=2)
add_table(doc, [
    ("Feature", "Mean", "Std Dev", "Min", "Median", "Max")] +
    [(col,
      f"{desc[col]['mean']:.3f}",
      f"{desc[col]['std']:.3f}",
      f"{desc[col]['min']:.3f}",
      f"{desc[col]['50%']:.3f}",
      f"{desc[col]['max']:.3f}") for col in numeric_cols
    ]
)

add_heading(doc, "2.5 — Class Balance", level=2)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.pie([no_cl, claims], labels=["No Claim","Claim"], colors=["#22c55e","#ef4444"],
       autopct="%1.1f%%", startangle=90, wedgeprops=dict(edgecolor="white",linewidth=2))
ax.set_title("Claim Outcome Distribution", fontweight="bold")
doc.add_picture(fig_buf(fig), width=Inches(3.2))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
plt.close()
add_caption(doc, "Fig 2 — 31.3% claim rate; stratified splits used to preserve this ratio")
add_callout(doc,
    "ROC-AUC (not accuracy) was used as the model selection criterion to avoid bias "
    "toward the majority No-Claim class in this moderately imbalanced dataset.", "CLASS IMBALANCE")

# ── 3. DATASET INSIGHTS & EDA ──────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "3. Dataset Insights & Exploratory Data Analysis")
add_para(doc,
    "EDA was performed across all 17 features. The charts below reveal the most significant "
    "claim risk signals in the dataset and directly support the feature importance rankings "
    "produced by the trained Gradient Boosting model.")

add_heading(doc, "3.1 — Claim Rate by Age Group", level=2)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(cr_age.index, cr_age.values, color="#2d6a9f", edgecolor="white")
ax.bar_label(bars, labels=[f"{v}%" for v in cr_age.values], padding=3, fontsize=9)
ax.set_title("Claim Rate by Age Group", fontweight="bold"); ax.set_ylabel("Claim Rate (%)")
ax.set_ylim(0, max(cr_age.values)+14); ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(5.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc, "Fig 3 — Claim rate drops sharply with age; 16-25 drivers are the highest-risk segment")
add_para(doc,
    f"Young drivers aged 16-25 have a claim rate of {cr_age.get('16-25','N/A')}% — "
    f"nearly 3x the rate of 65+ drivers ({cr_age.get('65+','N/A')}%). "
    "Inexperience, risk tolerance, and driving pattern differences are the primary drivers. "
    "This segment requires premium surcharges and graduated coverage structures.")

add_heading(doc, "3.2 — Claim Rate by Driving Experience", level=2)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(cr_exp.index, cr_exp.values, color="#f59e0b", edgecolor="white")
ax.bar_label(bars, labels=[f"{v}%" for v in cr_exp.values], padding=3, fontsize=9)
ax.set_title("Claim Rate by Driving Experience", fontweight="bold"); ax.set_ylabel("Claim Rate (%)")
ax.set_ylim(0, max(cr_exp.values)+14); ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(5.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc, "Fig 4 — Claim rate falls monotonically with driving experience")
add_para(doc,
    f"Novice drivers (0-9y) file claims at a rate of {cr_exp.get('0-9y','N/A')}% vs. "
    f"only {cr_exp.get('30y+','N/A')}% for 30+ year veterans. "
    "Driving experience is the single most discriminating demographic feature in the dataset "
    "and ranks as the top feature in the model's importance scores.")

add_heading(doc, "3.3 — Claim Rate by Income Class", level=2)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(cr_inc.index, cr_inc.values, color="#7c5cd8", edgecolor="white")
for i, v in enumerate(cr_inc.values): ax.text(v+0.5, i, f"{v}%", va="center", fontsize=9)
ax.set_title("Claim Rate by Income Class", fontweight="bold"); ax.set_xlabel("Claim Rate (%)")
ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(5.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc, "Fig 5 — Lower income classes have significantly higher claim rates")
add_para(doc,
    "Income class shows a strong monotonic relationship with claim probability. "
    "Poverty-level income correlates with older vehicles, deferred maintenance, higher financial stress, "
    "and a greater need to recoup accident costs through insurance — all contributing to higher claim frequency.")

add_heading(doc, "3.4 — Credit Score & Behavioural Signals", level=2)
# Credit score distribution
fig, ax = plt.subplots(figsize=(6, 4))
for outcome, color, label in [(0,"#22c55e","No Claim"),(1,"#ef4444","Claim")]:
    subset = df_raw[df_raw["OUTCOME"]==outcome]["CREDIT_SCORE"].dropna()
    ax.hist(subset, bins=30, alpha=0.65, color=color, label=label, edgecolor="white")
ax.set_title("Credit Score Distribution by Outcome", fontweight="bold")
ax.set_xlabel("Credit Score"); ax.set_ylabel("Count"); ax.legend()
ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(5.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc, "Fig 6 — Claim filers (red) tend to have lower credit scores than non-filers (green)")
no_claim_cs = round(df_raw[df_raw['OUTCOME']==0]['CREDIT_SCORE'].mean(), 3)
claim_cs    = round(df_raw[df_raw['OUTCOME']==1]['CREDIT_SCORE'].mean(), 3)
add_para(doc,
    f"No-claim policyholders have a mean credit score of {no_claim_cs} vs. "
    f"{claim_cs} for claim filers — a difference of {round(no_claim_cs-claim_cs,3)} points. "
    "Credit score acts as a proxy for financial discipline and risk aversion, making it the "
    "second most important feature in the model.")

# Speeding vs accidents scatter
fig, ax = plt.subplots(figsize=(6, 4))
for outcome, color, label in [(0,"#22c55e","No Claim"),(1,"#ef4444","Claim")]:
    sub = df_raw[df_raw["OUTCOME"]==outcome]
    ax.scatter(sub["SPEEDING_VIOLATIONS"], sub["PAST_ACCIDENTS"],
               alpha=0.2, color=color, label=label, s=15)
ax.set_xlabel("Speeding Violations"); ax.set_ylabel("Past Accidents")
ax.set_title("Speeding Violations vs Past Accidents", fontweight="bold"); ax.legend()
ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(5.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc, "Fig 7 — Claim filers cluster at higher speeding violations and past accident counts")

add_heading(doc, "3.5 — Vehicle Risk Signals", level=2)
fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(cr_veh.index, cr_veh.values, color=["#1e3a5f","#ef4444"], edgecolor="white", width=0.5)
ax.bar_label(bars, labels=[f"{v}%" for v in cr_veh.values], padding=3, fontsize=10)
ax.set_title("Claim Rate by Vehicle Type", fontweight="bold"); ax.set_ylabel("Claim Rate (%)")
ax.set_ylim(0, max(cr_veh.values)+12); ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(4.0))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc, "Fig 8 — Sports cars have a higher claim rate than sedans")
add_para(doc,
    f"Sports car drivers have a {cr_veh.get('sports car','N/A')}% claim rate vs. "
    f"{cr_veh.get('sedan','N/A')}% for sedan owners. "
    f"Vehicles made before 2015 have a {cr_yr.get('before 2015','N/A')}% claim rate vs. "
    f"{cr_yr.get('after 2015','N/A')}% for post-2015 models.")

# ── 4. MODEL INFORMATION ───────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "4. Model Information")

add_heading(doc, "4.1 — Algorithm: Gradient Boosting Classifier", level=2)
add_para(doc,
    "Gradient Boosting is a sequential ensemble method that builds decision trees one at a time, "
    "where each new tree corrects the residual errors of all previous trees. "
    "It optimises a differentiable loss function (log-loss for binary classification) "
    "using gradient descent in the space of functions — making it one of the most powerful "
    "off-the-shelf classifiers available.")
add_table(doc, [
    ("Hyperparameter",    "Value",    "Effect"),
    ("n_estimators",      "150",      "Number of boosting rounds; more trees = higher capacity but slower training"),
    ("learning_rate",     "0.1",      "Shrinkage applied to each tree's contribution; prevents overfitting"),
    ("max_depth",         "3",        "Controls individual tree complexity; shallow trees reduce variance"),
    ("random_state",      "42",       "Ensures full reproducibility of results"),
    ("subsample",         "1.0",      "Fraction of training samples used per tree (default = all)"),
    ("min_samples_split", "2",        "Minimum samples required to split a node"),
])

add_heading(doc, "4.2 — Model Selection Process", level=2)
add_para(doc,
    "Three classifiers were trained on an 80/20 stratified split. "
    "5-fold cross-validation ROC-AUC on the training set was used to rank models automatically. "
    "The best model was saved to model.pkl for serving by the Flask API.")
add_table(doc, [
    ("Model",                    "Test Accuracy", "CV-AUC (5-fold)", "Test AUC",              "Selected"),
    ("Logistic Regression",      "83.30%",        "0.9072",          "0.8925",                "—"),
    ("Random Forest (150 trees)","82.75%",        "0.9063",          "0.8919",                "—"),
    (f"{MODEL_NAME} (BEST)",     f"{round(MODEL_ACC*100,2)}%", "0.9239", f"{round(MODEL_AUC,4)}", "YES"),
])
add_callout(doc,
    f"{MODEL_NAME} was selected — highest test AUC ({round(MODEL_AUC,4)}) and CV-AUC (0.9239). "
    "The train-to-test AUC gap is only 0.0114, confirming minimal overfitting.", "SELECTION")

add_heading(doc, "4.3 — Preprocessing Pipeline", level=2)
add_table(doc, [
    ("Step",            "Method",                              "Applied To"),
    ("Null Imputation", "Median (numeric), Mode (categorical)","CREDIT_SCORE, ANNUAL_MILEAGE + any other nulls"),
    ("Feature Encoding","LabelEncoder (fitted on train data)", "AGE, GENDER, RACE, DRIVING_EXPERIENCE, EDUCATION, INCOME, VEHICLE_YEAR, VEHICLE_TYPE"),
    ("Train/Test Split","80/20 stratified (random_state=42)", "Full 10,000-record dataset"),
    ("Feature Scaling", "None required",                      "Tree-based models are invariant to feature scale"),
    ("ID Column",       "Dropped before training",            "ID carries no predictive signal"),
])

# ── 5. MODEL INSIGHTS & EVALUATION ────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "5. Model Insights & Evaluation")

add_heading(doc, "5.1 — Key Performance Metrics", level=2)
add_table(doc, [
    ("Metric",                  "Value",                                      "Interpretation"),
    ("Test Accuracy",           f"{round(MODEL_ACC*100,2)}%",                 "Overall correctness on held-out 2,000 records"),
    ("ROC-AUC",                 f"{round(MODEL_AUC,4)}",                      "Probability of ranking a claim above a non-claim"),
    ("No-Claim Precision",      f"{round(cr_dict['No Claim']['precision']*100,1)}%", "When model predicts no-claim, it's right this % of the time"),
    ("No-Claim Recall",         f"{round(cr_dict['No Claim']['recall']*100,1)}%",    "% of actual no-claims correctly identified"),
    ("Claim Precision",         f"{round(cr_dict['Claim']['precision']*100,1)}%",    "When model predicts claim, it's right this % of the time"),
    ("Claim Recall (Sensitivity)", f"{round(cr_dict['Claim']['recall']*100,1)}%",  "% of actual claims the model detected"),
    ("Specificity",             f"{specificity}%",                            "% of non-claims correctly cleared"),
    ("True Positives",          str(tp),                                      "Claims correctly predicted"),
    ("True Negatives",          str(tn),                                      "No-claims correctly cleared"),
    ("False Positives",         str(fp),                                      "Non-claims wrongly flagged (cost: unnecessary review)"),
    ("False Negatives",         str(fn),                                      "Claims missed (cost: unmanaged risk exposure)"),
])

add_heading(doc, "5.2 — Confusion Matrix", level=2)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_vals, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["No Claim","Claim"], yticklabels=["No Claim","Claim"])
ax.set_title(f"Confusion Matrix — {MODEL_NAME}", fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
doc.add_picture(fig_buf(fig), width=Inches(4.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc,
    f"Fig 9 — Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
add_para(doc,
    f"Of {int(cr_dict['Claim']['support'])} actual claims in the test set, "
    f"the model correctly identified {tp} ({round(cr_dict['Claim']['recall']*100,1)}% recall). "
    f"{fn} claims were missed (false negatives), representing undetected financial risk. "
    f"{fp} non-claim policyholders were incorrectly flagged (false positives), causing unnecessary intervention costs.")

add_heading(doc, "5.3 — ROC Curve", level=2)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, color="#2d6a9f", lw=2, label=f"ROC AUC = {roc_auc_val:.4f}")
ax.plot([0,1],[0,1],"--",color="#9ca3af",lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color="#2d6a9f")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", fontweight="bold"); ax.legend(); ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(4.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc,
    f"Fig 10 — ROC Curve: AUC = {round(roc_auc_val,4)}. "
    f"The model correctly ranks a random claim above a random non-claim {round(roc_auc_val*100,1)}% of the time.")

add_heading(doc, "5.4 — Feature Importance Analysis", level=2)
fi_top = fi_series.head(10).sort_values()
fig, ax = plt.subplots(figsize=(6, 5))
colors_fi = ["#1e3a5f" if v > fi_top.median() else "#2d6a9f" for v in fi_top.values]
ax.barh(fi_top.index, fi_top.values, color=colors_fi, edgecolor="white")
ax.set_title("Top 10 Feature Importances (Gradient Boosting)", fontweight="bold")
ax.set_xlabel("Importance Score"); ax.spines[["top","right"]].set_visible(False)
doc.add_picture(fig_buf(fig), width=Inches(5.5))
doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER; plt.close()
add_caption(doc, "Fig 11 — Top 10 feature importances; darker bars = above-median importance")
add_table(doc, [
    ("Rank", "Feature", "Importance", "Business Meaning")] +
    [(str(i+1), fi_series.index[i], f"{fi_series.iloc[i]:.4f}", desc_text)
     for i, desc_text in enumerate([
         "Primary behavioural risk indicator — years of safe driving correlates with fewer claims",
         "Financial reliability proxy — lower credit score = higher claim propensity",
         "Prior risk history — past accidents directly predict future claim likelihood",
         "Young drivers are highest-risk demographic",
         "Usage-based risk signal — more km = more exposure",
         "Violation history — each ticket increases predicted claim probability",
         "Financial stress indicator",
         "Vehicle risk profile — sports cars have higher accident rates",
         "Older vehicles lack modern safety features",
         "Geographic risk — postal code proxies for road quality and traffic density",
     ])
    ]
)

# ── 6. BUSINESS INSIGHTS ───────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "6. Business Insights")
add_para(doc,
    f"The model achieves ROC-AUC = {round(MODEL_AUC,4)}, meaning it correctly ranks a randomly "
    f"selected claim policyholder above a randomly selected non-claim policyholder "
    f"{round(MODEL_AUC*100,1)}% of the time — significantly better than random (50%). "
    "The following insights translate directly into underwriting and pricing decisions.")

add_heading(doc, "6.1 — High-Risk Segments", level=2)
add_table(doc, [
    ("Risk Signal",                  "Finding",                                      "Underwriting Action"),
    ("Age 16-25 + 0-9y experience",  f"Claim rate: {cr_age.get('16-25','N/A')}%",   "Premium surcharge + mandatory telematics"),
    ("2+ speeding violations",       "2x average claim rate",                        "Manual underwriting review required"),
    ("DUI offences > 0",             "Significant risk multiplier",                  "High-risk pool or coverage refusal"),
    ("Poverty income + low credit",  "Combined effect: ~2x average claim rate",      "Higher excess / deductible requirement"),
    ("Sports car + young driver",    "Compound risk: vehicle + demographic",          "Maximum surcharge tier"),
])

add_heading(doc, "6.2 — Low-Risk Competitive Opportunities", level=2)
add_table(doc, [
    ("Risk Signal",                  "Finding",                                      "Pricing Strategy"),
    ("30y+ driving experience",      f"Claim rate: only {cr_exp.get('30y+','N/A')}%","Preferred pricing + loyalty discount"),
    ("Upper-class income + 65+",     f"Claim rate: ~{cr_age.get('65+','N/A')}%",    "Lowest premium tier; competitive differentiator"),
    ("High credit score (>0.7)",     "Strong inverse correlation with claims",       "Credit-based discount where legally permitted"),
    ("0 violations + 0 accidents",   "Clean record: significant risk reduction",     "Safe driver discount (5-15%)"),
    ("Vehicle ownership + married",  "Combined responsibility indicators",           "Bundle discount for homeowner-drivers"),
])

add_heading(doc, "6.3 — Portfolio Risk Monitoring", level=2)
add_para(doc,
    "Running the model monthly across the full active policy book generates a portfolio risk "
    "distribution. Sudden shifts in aggregate predicted claim probability signal portfolio "
    "deterioration early — enabling reserve adjustments before claims materialise. "
    f"With {total:,} records, the model processes a prediction in milliseconds via the REST API, "
    "making real-time portfolio scoring computationally feasible.")

# ── 7. BUSINESS ANALYSIS ───────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "7. Business Analysis")

add_heading(doc, "7.1 — Problem Statement & Commercial Context", level=2)
add_para(doc,
    "Car insurance is a data-intensive, risk-priced product. Insurers that accurately predict "
    "claim likelihood gain three structural advantages: "
    "(1) more accurate actuarial pricing reduces adverse selection, "
    "(2) early flagging of high-risk policies enables proactive intervention, "
    "(3) portfolio-level risk scoring enables better capital allocation and reserving.")

add_heading(doc, "7.2 — Risk Segmentation Framework", level=2)
add_table(doc, [
    ("Segment",     "Typical Profile",                         "Predicted Probability", "Recommended Action"),
    ("HIGH RISK",   "Age 16-25, 0-9y exp, poverty, 2+ violations", "> 65%",            "Surcharge + telematics + manual review"),
    ("MEDIUM RISK", "Age 26-39, 10-19y, working class, 1 violation", "40% - 65%",      "Standard pricing + annual review"),
    ("LOW RISK",    "Age 40+, 20y+ exp, middle/upper, 0 violations", "< 40%",          "Preferred pricing + loyalty discount"),
])

add_heading(doc, "7.3 — Financial Impact on Test Portfolio", level=2)
add_table(doc, [
    ("Metric",                      "Value",  "Business Implication"),
    ("Actual claims in test set",   str(int(cr_dict['Claim']['support'])), "Baseline exposure without model"),
    ("Claims correctly flagged (TP)",str(tp), f"Early intervention possible for {tp} high-risk cases"),
    ("Claims missed (FN)",           str(fn), f"Unavoidable exposure — {fn} undetected claims"),
    ("False alarms (FP)",            str(fp), f"Unnecessary review cost for {fp} customers"),
    ("Correctly cleared (TN)",       str(tn), f"Efficient auto-approval for {tn} low-risk policies"),
    ("Claim detection rate",         f"{round(cr_dict['Claim']['recall']*100,1)}%",
                                              "Of every 100 actual claims, model flags ~76"),
])

add_heading(doc, "7.4 — Use-Case Applications", level=2)
add_table(doc, [
    ("Use Case",                     "How the Model is Used",                    "Expected Benefit"),
    ("Underwriting Decision Support","Query API with applicant profile → get risk score", "Reduce manual review workload by 70%+"),
    ("Dynamic Premium Pricing",      "Map probability (0-100%) to premium band", "More granular, actuarially fair pricing"),
    ("Claims Fraud Flagging",        "Flag anomaly when low-probability policy files claim", "Lightweight fraud indicator at zero additional cost"),
    ("Portfolio Risk Monitoring",    "Monthly batch scoring of full policy book", "Early warning of reserve shortfalls"),
    ("New Product Development",      "Identify low-risk segment for new product offerings", "Target preferred customers with competitive pricing"),
])

add_heading(doc, "7.5 — Model Limitations", level=2)
add_bullet(doc, [
    "Synthetic dataset: real-world distributions may differ significantly from this simulated data.",
    "Class imbalance (31.3% claims): claim precision is 74% — threshold tuning may be needed for production.",
    "Feature scope: no telematics, claim history, policy duration, or geographic risk scores.",
    "Static model: no automatic retraining — performance will degrade as driving patterns evolve.",
    "Fairness: RACE and GENDER features require regulatory review before use in premium pricing.",
    "Explainability: model lacks per-prediction SHAP explanations required for adverse action notices.",
])

# ── 8. UI: PREDICT PAGE ────────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "8. User Interface — Predict Page")
add_para(doc,
    "The Streamlit frontend runs on localhost:8501. The Predict page collects all 17 input "
    "features through a structured form, serialises them as JSON, and POSTs to the Flask API. "
    "The prediction result is rendered immediately below the form with full risk context.")

add_heading(doc, "8.1 — Input Form", level=2)
add_para(doc,
    "The form is divided into three collapsible sections: "
    "(1) Personal Information: age, gender, race, education, income, married, children, credit score — "
    "(2) Driving Profile: experience, speeding violations, DUI offences, past accidents, annual mileage — "
    "(3) Vehicle Details: type, year, ownership, postal code. "
    "The sidebar shows live Flask API connection status and model accuracy.")
add_screenshot(doc, "01_predict_form.png",
    "Screenshot 1 — Streamlit Predict Page: input form (localhost:8501)")

add_heading(doc, "8.2 — High-Risk Result: CLAIM Predicted", level=2)
add_para(doc,
    "When claim probability >= 65%, the result panel shows a red-bordered CLAIM card with: "
    "prediction label (CLAIM), claim probability %, High risk badge, and a red progress bar. "
    "Example profile: male, age 16-25, 0-9y experience, poverty income, credit 0.35, "
    "2 speeding violations, 1 past accident.")
add_screenshot(doc, "02_predict_result_claim.png",
    "Screenshot 2 — CLAIM result: 78.4% probability, High Risk (red panel)")

add_heading(doc, "8.3 — Low-Risk Result: NO CLAIM Predicted", level=2)
add_para(doc,
    "When claim probability < 40%, a green-bordered NO CLAIM card is shown with: "
    "NO CLAIM label, low probability %, Low risk badge, and a short green progress bar. "
    "Example profile: female, age 65+, 30y+ experience, upper-class income, credit 0.82, "
    "0 violations, 0 past accidents.")
add_screenshot(doc, "03_predict_result_no_claim.png",
    "Screenshot 3 — NO CLAIM result: 18.2% probability, Low Risk (green panel)")

# ── 9. UI: DASHBOARD ───────────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "9. User Interface — Analytics Dashboard")
add_para(doc,
    "The Dashboard page calls the Flask GET /stats endpoint at load time to fetch pre-computed "
    "dataset statistics and model metrics, then renders them as Matplotlib charts in Streamlit columns.")

add_heading(doc, "9.1 — KPI Row & Model Performance Card", level=2)
add_para(doc,
    f"Five st.metric cards span the full page width: Total Records ({total:,}), "
    f"Claims Filed ({claims:,} — {round(claims/total*100,1)}%), No Claims ({no_cl:,}), "
    f"Model Accuracy ({round(MODEL_ACC*100,2)}%), ROC-AUC ({round(MODEL_AUC,4)}). "
    f"Below the KPIs a model card shows the algorithm ({MODEL_NAME}), performance metric pills, "
    "and the full evaluation PNG (confusion matrix + feature importances) from training.")
add_screenshot(doc, "04_dashboard_kpi.png",
    "Screenshot 4 — Dashboard: KPI row, model card, and training evaluation plots")

add_heading(doc, "9.2 — EDA Charts Grid", level=2)
add_para(doc,
    "Four Matplotlib charts in a 2x2 grid: "
    "(1) Claims vs No Claims doughnut — 68.7% / 31.3% split; "
    "(2) Claim Rate by Age Group — 16-25 at ~52% vs 65+ at ~18%; "
    "(3) Claim Rate by Driving Experience — novice drivers file the most claims; "
    "(4) Income Class Distribution — dataset income composition. "
    "A scrollable raw data table (first 50 rows) appears below the charts.")
add_screenshot(doc, "05_dashboard_charts.png",
    "Screenshot 5 — Dashboard: EDA charts grid (claims, age, experience, income)")

# ── 10. SYSTEM ARCHITECTURE ────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "10. System Architecture")
add_callout(doc,
    "Streamlit (port 8501) → HTTP POST/GET → Flask REST API (port 5000) → model.pkl. "
    "Frontend and backend are fully decoupled — any HTTP client can call the API directly.", "ARCHITECTURE")
add_table(doc, [
    ("File",                    "Role",                                         "Key Libraries"),
    ("train_model.py",          "Training pipeline — data, 3 models, model.pkl","pandas, scikit-learn, matplotlib"),
    ("app.py",                  "Flask REST API — POST /predict, GET /stats",   "flask, flask-cors, numpy"),
    ("streamlit_app.py",        "Streamlit UI — Predict + Dashboard pages",     "streamlit, requests, matplotlib"),
    ("generate_screenshots.py", "Renders all 6 UI mockup PNGs",                "matplotlib, pandas, pickle"),
    ("generate_report.py",      "Self-contained HTML report (base64 images)",   "matplotlib, seaborn, sklearn"),
    ("generate_docx.py",        "This Word document report",                    "python-docx, matplotlib, sklearn"),
    ("model.pkl",               "Serialised: model + encoders + metadata",      "pickle"),
    ("requirements.txt",        "All Python package pins",                      "—"),
])

# ── 11. HOW TO RUN ─────────────────────────────────────────────────────────────
doc.add_paragraph()
add_heading(doc, "11. How to Run")
add_table(doc, [
    ("Step", "Terminal", "Command",                              "Result"),
    ("1",    "Any",      "pip install -r requirements.txt",     "All dependencies installed"),
    ("2",    "Any",      "python train_model.py",               "model.pkl + evaluation plot created"),
    ("3",    "A",        "python app.py",                       "Flask API on http://127.0.0.1:5000"),
    ("4",    "B",        "streamlit run streamlit_app.py",      "Streamlit UI on http://localhost:8501"),
    ("5",    "Any",      "python generate_screenshots.py",      "6 PNG mockups in static/screenshots/"),
    ("6",    "Any",      "python generate_report.py",           "project_report.html created"),
    ("7",    "Any",      "python generate_docx.py",             "project_report.docx created"),
])
add_callout(doc,
    "Flask API (Step 3) must be running BEFORE opening the Streamlit UI (Step 4). "
    "Both terminals must remain open simultaneously during use.", "IMPORTANT")

# ── 12. FUTURE SCOPE ───────────────────────────────────────────────────────────
doc.add_page_break()
add_heading(doc, "12. Future Scope")
add_para(doc,
    "The current system provides a solid production-grade baseline. "
    "The following improvements are prioritised by expected impact and implementation effort.")

add_heading(doc, "12.1 — Model Improvements", level=2)
add_table(doc, [
    ("Enhancement",                 "Description",                                              "Expected Impact"),
    ("XGBoost / LightGBM",          "Replace GBM with native categorical support + GPU speed",  "AUC +0.01 to +0.03"),
    ("Hyperparameter Tuning",       "Optuna Bayesian search for learning_rate, depth, trees",   "AUC +0.005 to +0.02"),
    ("SMOTE Oversampling",          "Synthetic minority oversampling to address class imbalance","Claim recall +5-8%"),
    ("Threshold Optimisation",      "Precision-Recall curve analysis to find optimal threshold", "Better FP/FN trade-off"),
    ("Stacked Ensemble",            "Meta-learner combining all 3 base models",                  "AUC +0.01 to +0.02"),
])

add_heading(doc, "12.2 — Feature Engineering", level=2)
add_table(doc, [
    ("Feature Addition",            "Description",                                              "Expected Impact"),
    ("Telematics / UBI Data",       "Hard braking, night driving %, average speed from IoT",    "AUC +0.03 to +0.08"),
    ("Geospatial Risk Scores",      "Replace postal code with accident frequency per km2",       "AUC +0.01 to +0.03"),
    ("Policy History Features",     "Prior claims, years as customer, payment punctuality",      "AUC +0.02 to +0.04"),
    ("Claim Severity Regression",   "Two-stage: predict claim likelihood, then claim amount",    "Enables expected-loss pricing"),
    ("Weather / Road Risk Index",   "External data: seasonal accident rates by location",        "Regional pricing granularity"),
])

add_heading(doc, "12.3 — System & Infrastructure", level=2)
add_table(doc, [
    ("Improvement",                 "Description",                                              "Benefit"),
    ("Docker Containerisation",     "Flask + Streamlit as separate Docker containers",           "One-command cloud deployment"),
    ("MLflow Experiment Tracking",  "Model versioning, metric logging, model registry",          "Full MLOps lifecycle management"),
    ("API Authentication",          "JWT-based auth + rate limiting for Flask API",              "Production security baseline"),
    ("Scheduled Retraining",        "Drift detection (Evidently AI) + auto-retrain trigger",     "Model freshness in production"),
    ("Grafana Monitoring",          "Real-time API latency, prediction distribution tracking",   "Production observability"),
])

add_heading(doc, "12.4 — Explainability & Fairness", level=2)
add_table(doc, [
    ("Initiative",                  "Description",                                              "Compliance Relevance"),
    ("SHAP Explanations",           "Per-prediction feature attribution in API response",        "FCRA / ECOA adverse action notices"),
    ("IBM AI Fairness 360 Audit",   "Disparate impact analysis across RACE, GENDER",            "Insurance regulatory compliance"),
    ("LIME Integration",            "Local interpretable model-agnostic explanations",           "Customer-facing explanation reports"),
    ("Model Card Documentation",    "Standardised model card with limitations and bias analysis","EU AI Act / governance requirements"),
])

add_heading(doc, "12.5 — Recommended Next Steps (Priority Order)", level=2)
add_bullet(doc, [
    "Step 1 — Threshold optimisation: tune classification threshold for optimal FN/FP trade-off",
    "Step 2 — SHAP integration: add per-prediction explanations to Flask API response",
    "Step 3 — LightGBM replacement: faster training + native categorical encoding + higher AUC",
    "Step 4 — Docker deployment: containerise for scalable cloud hosting",
    "Step 5 — Telematics feature ingestion: most impactful accuracy uplift possible",
])

# ── FOOTER ─────────────────────────────────────────────────────────────────────
doc.add_paragraph()
foot = doc.add_paragraph(
    "Car Insurance Claim Prediction  |  IBM AICTE AI/ML Internship Project  |  Made with IBM Bob")
foot.alignment = WD_ALIGN_PARAGRAPH.CENTER
foot.runs[0].font.color.rgb = RGBColor(0x9C, 0xA3, 0xAF); foot.runs[0].font.size = Pt(9)

# ── Save ───────────────────────────────────────────────────────────────────────
out_path = os.path.join(BASE_DIR, "project_report.docx")
doc.save(out_path)
print(f"DOCX report saved -> {out_path}  ({os.path.getsize(out_path)//1024} KB)")


### Generate Report - HTML - generate_report.py

In [ ]:
"""
generate_report.py
==================
Generates a fully comprehensive self-contained HTML project report.
Sections: Dataset Info, Dataset Insights, Model Info, Model Insights,
Business Insights, Business Analysis, UI Screenshots, Future Scope.

Run:
    python generate_report.py
    # outputs: project_report.html
"""

import os, base64, pickle, io, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc as sk_auc)
from sklearn.model_selection import train_test_split

BASE_DIR    = os.path.dirname(os.path.abspath(__file__))
SCREENS_DIR = os.path.join(BASE_DIR, "static", "screenshots")

# ── Load artefacts ─────────────────────────────────────────────────────────────
with open(os.path.join(BASE_DIR, "model.pkl"), "rb") as f:
    art = pickle.load(f)
MODEL, ENCODERS = art["model"], art["encoders"]
FEATURE_NAMES   = art["feature_names"]
MODEL_NAME      = art["model_name"]
MODEL_ACC       = art["accuracy"]
MODEL_AUC       = art["auc"]

# ── Load & prepare data ────────────────────────────────────────────────────────
df_raw = pd.read_csv(os.path.join(BASE_DIR, "Car_Insurance_Claim.csv"))
df = df_raw.copy()
df.drop(columns=["ID"], inplace=True)
for col in df.columns:
    if df[col].dtype == "object":
        df[col].fillna(df[col].mode()[0], inplace=True)
    else:
        df[col].fillna(df[col].median(), inplace=True)

CAT_COLS = ["AGE","GENDER","RACE","DRIVING_EXPERIENCE","EDUCATION","INCOME","VEHICLE_YEAR","VEHICLE_TYPE"]
for col in CAT_COLS:
    df[col] = ENCODERS[col].transform(df[col].astype(str))

X = df.drop(columns=["OUTCOME"])
y = df["OUTCOME"].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_pred = MODEL.predict(X_te)
y_prob = MODEL.predict_proba(X_te)[:, 1]

total  = len(df_raw)
claims = int(df_raw["OUTCOME"].sum())
no_cl  = total - claims

# ── Pre-compute dataset analytics ─────────────────────────────────────────────
cr_age  = df_raw.groupby("AGE")["OUTCOME"].mean().mul(100).round(1)
cr_exp  = df_raw.groupby("DRIVING_EXPERIENCE")["OUTCOME"].mean().mul(100).round(1)
cr_inc  = df_raw.groupby("INCOME")["OUTCOME"].mean().mul(100).round(1).sort_values(ascending=False)
cr_gen  = df_raw.groupby("GENDER")["OUTCOME"].mean().mul(100).round(1)
cr_edu  = df_raw.groupby("EDUCATION")["OUTCOME"].mean().mul(100).round(1)
cr_veh  = df_raw.groupby("VEHICLE_TYPE")["OUTCOME"].mean().mul(100).round(1)
cr_yr   = df_raw.groupby("VEHICLE_YEAR")["OUTCOME"].mean().mul(100).round(1)

num_nulls   = df_raw.isnull().sum()
total_nulls = int(num_nulls.sum())
null_cols   = num_nulls[num_nulls > 0].to_dict()

numeric_cols = ["CREDIT_SCORE","ANNUAL_MILEAGE","SPEEDING_VIOLATIONS","DUIS","PAST_ACCIDENTS"]
desc = df_raw[numeric_cols].describe().round(3)

cr_dict = classification_report(y_te, y_pred, target_names=["No Claim","Claim"], output_dict=True)
cm_vals = confusion_matrix(y_te, y_pred)
tn, fp, fn, tp = cm_vals.ravel()
specificity = round(tn / (tn + fp) * 100, 1)
fi_series   = pd.Series(MODEL.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)

# ── Helpers ────────────────────────────────────────────────────────────────────
def file_b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def fig_b64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

# ── Load screenshots ───────────────────────────────────────────────────────────
sc = {k: file_b64(os.path.join(SCREENS_DIR, v)) for k, v in {
    "predict_form":     "01_predict_form.png",
    "predict_claim":    "02_predict_result_claim.png",
    "predict_no_claim": "03_predict_result_no_claim.png",
    "dashboard_kpi":    "04_dashboard_kpi.png",
    "dashboard_charts": "05_dashboard_charts.png",
    "dataset_preview":  "06_dataset_preview.png",
}.items()}

# ── Generate charts ────────────────────────────────────────────────────────────
# 1. Outcome pie
fig, ax = plt.subplots(figsize=(4.5,4.5))
ax.pie([no_cl, claims], labels=["No Claim","Claim"], colors=["#22c55e","#ef4444"],
       autopct="%1.1f%%", startangle=90, wedgeprops=dict(edgecolor="white",linewidth=2))
ax.set_title("Claim Outcome Distribution", fontweight="bold")
b64_pie = fig_b64(fig); plt.close()

# 2. Claim rate by age
fig, ax = plt.subplots(figsize=(5,4))
bars = ax.bar(cr_age.index, cr_age.values, color="#2d6a9f", edgecolor="white")
ax.bar_label(bars, labels=[f"{v}%" for v in cr_age.values], padding=3, fontsize=9)
ax.set_title("Claim Rate by Age Group", fontweight="bold"); ax.set_ylabel("Claim Rate (%)")
ax.set_ylim(0, max(cr_age.values)+14); ax.spines[["top","right"]].set_visible(False)
b64_age = fig_b64(fig); plt.close()

# 3. Claim rate by driving experience
fig, ax = plt.subplots(figsize=(5,4))
bars = ax.bar(cr_exp.index, cr_exp.values, color="#f59e0b", edgecolor="white")
ax.bar_label(bars, labels=[f"{v}%" for v in cr_exp.values], padding=3, fontsize=9)
ax.set_title("Claim Rate by Driving Experience", fontweight="bold"); ax.set_ylabel("Claim Rate (%)")
ax.set_ylim(0, max(cr_exp.values)+14); ax.spines[["top","right"]].set_visible(False)
b64_exp = fig_b64(fig); plt.close()

# 4. Claim rate by income
fig, ax = plt.subplots(figsize=(5,4))
ax.barh(cr_inc.index, cr_inc.values, color="#7c5cd8", edgecolor="white")
for i,v in enumerate(cr_inc.values): ax.text(v+0.5,i,f"{v}%",va="center",fontsize=9)
ax.set_title("Claim Rate by Income Class", fontweight="bold"); ax.set_xlabel("Claim Rate (%)")
ax.spines[["top","right"]].set_visible(False)
b64_inc = fig_b64(fig); plt.close()

# 5. Claim rate by gender
fig, ax = plt.subplots(figsize=(4,3.5))
colors_g = ["#2d6a9f","#f59e0b"]
bars = ax.bar(cr_gen.index, cr_gen.values, color=colors_g, edgecolor="white", width=0.5)
ax.bar_label(bars, labels=[f"{v}%" for v in cr_gen.values], padding=3, fontsize=10)
ax.set_title("Claim Rate by Gender", fontweight="bold"); ax.set_ylabel("Claim Rate (%)")
ax.set_ylim(0, max(cr_gen.values)+12); ax.spines[["top","right"]].set_visible(False)
b64_gen = fig_b64(fig); plt.close()

# 6. Claim rate by vehicle type
fig, ax = plt.subplots(figsize=(4,3.5))
bars = ax.bar(cr_veh.index, cr_veh.values, color=["#1e3a5f","#ef4444"], edgecolor="white", width=0.5)
ax.bar_label(bars, labels=[f"{v}%" for v in cr_veh.values], padding=3, fontsize=10)
ax.set_title("Claim Rate by Vehicle Type", fontweight="bold"); ax.set_ylabel("Claim Rate (%)")
ax.set_ylim(0, max(cr_veh.values)+12); ax.spines[["top","right"]].set_visible(False)
b64_veh = fig_b64(fig); plt.close()

# 7. Confusion matrix
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm_vals, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["No Claim","Claim"], yticklabels=["No Claim","Claim"])
ax.set_title(f"Confusion Matrix — {MODEL_NAME}", fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
b64_cm = fig_b64(fig); plt.close()

# 8. Feature importance (top 10)
fi_top = fi_series.head(10).sort_values()
fig, ax = plt.subplots(figsize=(6,5))
colors_fi = ["#1e3a5f" if v > fi_top.median() else "#2d6a9f" for v in fi_top.values]
ax.barh(fi_top.index, fi_top.values, color=colors_fi, edgecolor="white")
ax.set_title("Top 10 Feature Importances", fontweight="bold"); ax.set_xlabel("Importance Score")
ax.spines[["top","right"]].set_visible(False)
b64_fi = fig_b64(fig); plt.close()

# 9. ROC curve
fpr, tpr, _ = roc_curve(y_te, y_prob)
roc_auc_val = sk_auc(fpr, tpr)
fig, ax = plt.subplots(figsize=(5,4))
ax.plot(fpr, tpr, color="#2d6a9f", lw=2, label=f"ROC (AUC = {roc_auc_val:.4f})")
ax.plot([0,1],[0,1],"--",color="#9ca3af",lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color="#2d6a9f")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", fontweight="bold"); ax.legend(fontsize=9)
ax.spines[["top","right"]].set_visible(False)
b64_roc = fig_b64(fig); plt.close()

# 10. Credit score distribution by outcome
fig, ax = plt.subplots(figsize=(5,4))
df_raw.groupby("OUTCOME")["CREDIT_SCORE"].plot(kind="hist", ax=ax, alpha=0.65, bins=30,
    color=["#22c55e","#ef4444"] if False else None)
for outcome, color, label in [(0,"#22c55e","No Claim"),(1,"#ef4444","Claim")]:
    subset = df_raw[df_raw["OUTCOME"]==outcome]["CREDIT_SCORE"].dropna()
    ax.hist(subset, bins=30, alpha=0.65, color=color, label=label, edgecolor="white")
ax.set_title("Credit Score Distribution by Outcome", fontweight="bold")
ax.set_xlabel("Credit Score"); ax.set_ylabel("Count"); ax.legend()
ax.spines[["top","right"]].set_visible(False)
b64_credit = fig_b64(fig); plt.close()

# 11. Speeding violations vs past accidents scatter
fig, ax = plt.subplots(figsize=(5,4))
for outcome, color, label in [(0,"#22c55e","No Claim"),(1,"#ef4444","Claim")]:
    sub = df_raw[df_raw["OUTCOME"]==outcome]
    ax.scatter(sub["SPEEDING_VIOLATIONS"], sub["PAST_ACCIDENTS"],
               alpha=0.25, color=color, label=label, s=18)
ax.set_xlabel("Speeding Violations"); ax.set_ylabel("Past Accidents")
ax.set_title("Speeding Violations vs Past Accidents", fontweight="bold"); ax.legend()
ax.spines[["top","right"]].set_visible(False)
b64_scatter = fig_b64(fig); plt.close()

# ── Classification report table HTML ──────────────────────────────────────────
def cr_table_html():
    rows = ""
    for lbl in ["No Claim","Claim","macro avg","weighted avg"]:
        d = cr_dict.get(lbl, {})
        rows += (f"<tr><td>{lbl}</td><td>{d['precision']:.4f}</td>"
                 f"<td>{d['recall']:.4f}</td><td>{d['f1-score']:.4f}</td>"
                 f"<td>{int(d.get('support',0))}</td></tr>")
    return rows

# ── Numeric stats table HTML ───────────────────────────────────────────────────
def num_stats_html():
    rows = ""
    for col in numeric_cols:
        s = desc[col]
        rows += (f"<tr><td>{col}</td><td>{s['mean']:.3f}</td><td>{s['std']:.3f}</td>"
                 f"<td>{s['min']:.3f}</td><td>{s['50%']:.3f}</td><td>{s['max']:.3f}</td></tr>")
    return rows

# ══════════════════════════════════════════════════════════════════════════════
# CSS
# ══════════════════════════════════════════════════════════════════════════════
CSS = """
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,"Segoe UI",system-ui,sans-serif;
     background:#f0f4f8;color:#1f2328;font-size:14px;line-height:1.7}

.cover{background:linear-gradient(135deg,#1e3a5f,#2d6a9f);
       color:#fff;text-align:center;padding:4rem 2rem 3rem}
.cover h1{font-size:2.4rem;font-weight:800;margin-bottom:.6rem}
.cover .sub{font-size:1rem;opacity:.85;max-width:640px;margin:.6rem auto 0}
.cover .meta{margin-top:1.5rem;font-size:.82rem;opacity:.7;line-height:2}

.toc{background:#fff;border-radius:10px;padding:1.4rem 2rem;
     box-shadow:0 2px 10px rgba(0,0,0,.07);margin:2rem auto;max-width:960px}
.toc h2{font-size:1rem;color:#1e3a5f;font-weight:800;margin-bottom:.8rem}
.toc-grid{display:grid;grid-template-columns:1fr 1fr;gap:.2rem .5rem}
.toc ol{padding-left:1.2rem}
.toc li{margin:.22rem 0}.toc a{color:#2d6a9f;text-decoration:none}
.toc a:hover{text-decoration:underline}

.wrapper{max-width:960px;margin:0 auto;padding:0 1.2rem 4rem}
.section{background:#fff;border-radius:12px;
         box-shadow:0 2px 12px rgba(0,0,0,.07);
         padding:1.8rem 2rem;margin-bottom:2rem}
.section h2{font-size:1.2rem;color:#1e3a5f;font-weight:800;
            padding-bottom:.6rem;border-bottom:2px solid #e5e7eb;margin-bottom:1.2rem}
.section h3{font-size:1rem;color:#374151;font-weight:700;margin:1.4rem 0 .6rem}
p{margin-bottom:.8rem}
ul,ol{margin:.4rem 0 .8rem 1.4rem} li{margin:.3rem 0}

.kpi-row{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));
         gap:1rem;margin-bottom:1.4rem}
.kpi{background:#f7f8fa;border-radius:8px;padding:.9rem 1rem;
     border-left:4px solid #2d6a9f;text-align:center}
.kpi.green{border-left-color:#22c55e}.kpi.red{border-left-color:#ef4444}
.kpi.purple{border-left-color:#7c5cd8}.kpi.amber{border-left-color:#f59e0b}
.kpi.teal{border-left-color:#06b6d4}
.kpi-val{font-size:1.65rem;font-weight:800;color:#1e3a5f}
.kpi-lbl{font-size:.7rem;color:#6b7280;text-transform:uppercase;letter-spacing:.5px}

.chart-grid{display:grid;grid-template-columns:1fr 1fr;gap:1.4rem;margin-top:1rem}
.chart-grid-3{display:grid;grid-template-columns:1fr 1fr 1fr;gap:1.2rem;margin-top:1rem}
.chart-box{text-align:center}
.chart-box img{width:100%;border-radius:8px;border:1px solid #e5e7eb}
.chart-box .cap{font-size:.78rem;color:#6b7280;margin-top:.4rem;font-style:italic}

.insight-grid{display:grid;grid-template-columns:1fr 1fr;gap:1rem;margin-top:1rem}
.insight-card{background:#f7f8fa;border-radius:8px;padding:1rem 1.2rem;
              border-left:4px solid #2d6a9f}
.insight-card.green{border-left-color:#22c55e}
.insight-card.red{border-left-color:#ef4444}
.insight-card.amber{border-left-color:#f59e0b}
.insight-card.purple{border-left-color:#7c5cd8}
.insight-card h4{font-size:.88rem;font-weight:700;color:#1e3a5f;margin-bottom:.4rem}
.insight-card p{font-size:.84rem;margin:0;color:#374151}

.callout{border-radius:8px;padding:1rem 1.3rem;margin:1rem 0;font-size:.9rem}
.callout.blue{background:#eff6ff;border-left:4px solid #2d6a9f;color:#1e40af}
.callout.green{background:#f0fdf4;border-left:4px solid #22c55e;color:#166534}
.callout.amber{background:#fffbeb;border-left:4px solid #f59e0b;color:#92400e}
.callout.red{background:#fef2f2;border-left:4px solid #ef4444;color:#991b1b}

.screenshot-block{margin:1.4rem 0}
.screenshot-block img{width:100%;border-radius:10px;
  border:2px solid #e5e7eb;box-shadow:0 4px 20px rgba(0,0,0,.12)}
.screenshot-block .sc-caption{background:#f7f8fa;border-radius:0 0 10px 10px;
  padding:.8rem 1.2rem;border:2px solid #e5e7eb;border-top:none;
  font-size:.86rem;color:#374151;line-height:1.6}
.sc-label{display:inline-block;background:#1e3a5f;color:#fff;
  font-size:.7rem;font-weight:700;padding:.2rem .7rem;
  border-radius:999px;margin-bottom:.5rem;letter-spacing:.4px}

table{width:100%;border-collapse:collapse;font-size:.88rem;margin-top:.6rem}
th{background:#1e3a5f;color:#fff;padding:.55rem .85rem;text-align:left;font-size:.8rem}
td{padding:.48rem .85rem;border-bottom:1px solid #e5e7eb}
tr:nth-child(even) td{background:#f7f8fa}

.pipeline{display:flex;gap:.5rem;flex-wrap:wrap;margin-top:.8rem}
.step{background:#1e3a5f;color:#fff;padding:.4rem .9rem;
      border-radius:999px;font-size:.8rem;font-weight:600}
.arrow{display:flex;align-items:center;color:#9ca3af;font-size:1rem}

.tag{display:inline-block;padding:.15rem .6rem;border-radius:999px;
     font-size:.75rem;font-weight:700;margin:.15rem}
.tag-blue{background:#dbeafe;color:#1e40af}
.tag-green{background:#dcfce7;color:#166534}
.tag-red{background:#fee2e2;color:#991b1b}
.tag-amber{background:#fef3c7;color:#92400e}
.tag-purple{background:#ede9fe;color:#5b21b6}

footer{text-align:center;padding:1.2rem;font-size:.75rem;color:#9ca3af;
       border-top:1px solid #e5e7eb;margin-top:3rem}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML — PART 1: HEAD + COVER + TOC + SECTIONS 1-3
# ══════════════════════════════════════════════════════════════════════════════
html_part1 = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<title>Car Insurance Claim Prediction — Full Project Report</title>
<style>{CSS}</style>
</head>
<body>

<div class="cover">
  <h1>Car Insurance Claim Prediction</h1>
  <div class="sub">A comprehensive end-to-end machine learning project for predicting whether
  a car insurance policyholder will file a claim — built with Python, Flask &amp; Streamlit.</div>
  <div class="meta">
    IBM AICTE AI/ML Internship Project &nbsp;|&nbsp; Dataset: {total:,} records · 18 features<br/>
    Best Model: <strong>{MODEL_NAME}</strong> &nbsp;|&nbsp;
    Accuracy: <strong>{round(MODEL_ACC*100,2)}%</strong> &nbsp;|&nbsp;
    ROC-AUC: <strong>{round(MODEL_AUC,4)}</strong>
  </div>
</div>

<div class="toc">
  <h2>Table of Contents</h2>
  <div class="toc-grid">
    <ol>
      <li><a href="#s1">Project Overview</a></li>
      <li><a href="#s2">Dataset Information</a></li>
      <li><a href="#s3">Dataset Insights &amp; EDA</a></li>
      <li><a href="#s4">Model Information</a></li>
      <li><a href="#s5">Model Insights &amp; Evaluation</a></li>
      <li><a href="#s6">Business Insights</a></li>
    </ol>
    <ol start="7">
      <li><a href="#s7">Business Analysis</a></li>
      <li><a href="#s8">User Interface — Predict Page</a></li>
      <li><a href="#s9">User Interface — Analytics Dashboard</a></li>
      <li><a href="#s10">System Architecture</a></li>
      <li><a href="#s11">How to Run</a></li>
      <li><a href="#s12">Future Scope</a></li>
    </ol>
  </div>
</div>

<div class="wrapper">

<!-- ══ 1. PROJECT OVERVIEW ══ -->
<div class="section" id="s1">
  <h2>1. Project Overview</h2>
  <p>Car insurance companies face significant financial risk from fraudulent or high-frequency
  claims. Predicting <em>whether</em> a policyholder will file a claim — before it happens —
  allows insurers to price risk accurately, prioritise underwriting reviews, and intervene
  proactively with high-risk customers.</p>
  <p>This project builds a production-ready binary classification pipeline that takes
  17 policyholder attributes and outputs a claim probability, a prediction label, and a
  risk tier (Low / Medium / High). The system consists of three independent layers:</p>
  <ul>
    <li><strong>train_model.py</strong> — data ingestion, preprocessing, multi-model training, auto-selection, artefact export</li>
    <li><strong>app.py</strong> — Flask REST API exposing <code>POST /predict</code> and <code>GET /stats</code></li>
    <li><strong>streamlit_app.py</strong> — Streamlit web UI with a Predict page and an Analytics Dashboard</li>
  </ul>
  <h3>ML Pipeline</h3>
  <div class="pipeline">
    <div class="step">Data Loading</div><div class="arrow">→</div>
    <div class="step">EDA</div><div class="arrow">→</div>
    <div class="step">Preprocessing</div><div class="arrow">→</div>
    <div class="step">Label Encoding</div><div class="arrow">→</div>
    <div class="step">Train / Test Split</div><div class="arrow">→</div>
    <div class="step">3-Model Training</div><div class="arrow">→</div>
    <div class="step">CV Evaluation</div><div class="arrow">→</div>
    <div class="step">Auto-Select Best</div><div class="arrow">→</div>
    <div class="step">Flask API</div><div class="arrow">→</div>
    <div class="step">Streamlit UI</div>
  </div>
  <h3>Technology Stack</h3>
  <table>
    <tr><th>Layer</th><th>Technology</th><th>Purpose</th></tr>
    <tr><td>ML Modelling</td><td>scikit-learn</td><td>Random Forest, Gradient Boosting, Logistic Regression + CV</td></tr>
    <tr><td>Backend API</td><td>Flask + flask-cors</td><td>REST API — JSON prediction service on port 5000</td></tr>
    <tr><td>Frontend UI</td><td>Streamlit</td><td>Interactive web app — Predict &amp; Dashboard pages on port 8501</td></tr>
    <tr><td>Data Processing</td><td>pandas, NumPy</td><td>ETL, feature engineering, null imputation</td></tr>
    <tr><td>Visualisation</td><td>Matplotlib, Seaborn</td><td>EDA charts, model evaluation plots, UI mockups</td></tr>
    <tr><td>Report Generation</td><td>Python (this file)</td><td>Self-contained HTML report with all charts embedded</td></tr>
    <tr><td>Document Export</td><td>python-docx</td><td>Word document report with screenshots and tables</td></tr>
  </table>
</div>

<!-- ══ 2. DATASET INFORMATION ══ -->
<div class="section" id="s2">
  <h2>2. Dataset Information</h2>
  <div class="kpi-row">
    <div class="kpi green"><div class="kpi-val">{total:,}</div><div class="kpi-lbl">Total Records</div></div>
    <div class="kpi"><div class="kpi-val">17</div><div class="kpi-lbl">Input Features</div></div>
    <div class="kpi red"><div class="kpi-val">{claims:,}</div><div class="kpi-lbl">Claims Filed (=1)</div></div>
    <div class="kpi green"><div class="kpi-val">{no_cl:,}</div><div class="kpi-lbl">No Claims (=0)</div></div>
    <div class="kpi amber"><div class="kpi-val">{round(claims/total*100,1)}%</div><div class="kpi-lbl">Claim Rate</div></div>
    <div class="kpi teal"><div class="kpi-val">{total_nulls}</div><div class="kpi-lbl">Missing Values</div></div>
  </div>

  <div class="callout blue">
    <strong>Source:</strong> Car_Insurance_Claim.csv — synthetic insurance dataset with {total:,}
    policyholder records, 8 categorical features, 9 numeric/binary features, and 1 binary target
    column (OUTCOME). Missing values exist in CREDIT_SCORE, ANNUAL_MILEAGE (imputed at runtime).
  </div>

  <h3>Dataset Preview</h3>
  <div class="screenshot-block">
    <img src="data:image/png;base64,{sc['dataset_preview']}" alt="Dataset Preview"/>
    <div class="sc-caption">
      <strong>First 8 rows of Car_Insurance_Claim.csv.</strong>
      OUTCOME is colour-coded: red = claim filed (1.0), green = no claim (0.0).
      Notice the mix of categorical (AGE, GENDER, INCOME) and numeric (CREDIT_SCORE, ANNUAL_MILEAGE) columns.
    </div>
  </div>

  <h3>Feature Dictionary</h3>
  <table>
    <tr><th>Feature</th><th>Type</th><th>Values / Range</th><th>Description</th></tr>
    <tr><td>AGE</td><td><span class="tag tag-amber">Categorical</span></td><td>16-25, 26-39, 40-64, 65+</td><td>Driver age group</td></tr>
    <tr><td>GENDER</td><td><span class="tag tag-amber">Categorical</span></td><td>male, female</td><td>Driver gender</td></tr>
    <tr><td>RACE</td><td><span class="tag tag-amber">Categorical</span></td><td>majority, minority</td><td>Race category</td></tr>
    <tr><td>DRIVING_EXPERIENCE</td><td><span class="tag tag-amber">Categorical</span></td><td>0-9y, 10-19y, 20-29y, 30y+</td><td>Years of driving experience</td></tr>
    <tr><td>EDUCATION</td><td><span class="tag tag-amber">Categorical</span></td><td>none, high school, university</td><td>Highest education level</td></tr>
    <tr><td>INCOME</td><td><span class="tag tag-amber">Categorical</span></td><td>poverty, working class, middle class, upper class</td><td>Income bracket</td></tr>
    <tr><td>VEHICLE_YEAR</td><td><span class="tag tag-amber">Categorical</span></td><td>before 2015, after 2015</td><td>Vehicle manufacture year band</td></tr>
    <tr><td>VEHICLE_TYPE</td><td><span class="tag tag-amber">Categorical</span></td><td>sedan, sports car</td><td>Type of vehicle insured</td></tr>
    <tr><td>CREDIT_SCORE</td><td><span class="tag tag-blue">Numeric</span></td><td>0.0 – 1.0</td><td>Normalised credit score</td></tr>
    <tr><td>ANNUAL_MILEAGE</td><td><span class="tag tag-blue">Numeric</span></td><td>~5,000 – 25,000 km</td><td>Kilometres driven per year</td></tr>
    <tr><td>SPEEDING_VIOLATIONS</td><td><span class="tag tag-blue">Numeric</span></td><td>0 – 20+</td><td>Number of speeding tickets</td></tr>
    <tr><td>DUIS</td><td><span class="tag tag-blue">Numeric</span></td><td>0 – 5</td><td>DUI offences recorded</td></tr>
    <tr><td>PAST_ACCIDENTS</td><td><span class="tag tag-blue">Numeric</span></td><td>0 – 15</td><td>Prior accident count</td></tr>
    <tr><td>VEHICLE_OWNERSHIP</td><td><span class="tag tag-green">Binary</span></td><td>0, 1</td><td>Whether driver owns the vehicle</td></tr>
    <tr><td>MARRIED</td><td><span class="tag tag-green">Binary</span></td><td>0, 1</td><td>Marital status</td></tr>
    <tr><td>CHILDREN</td><td><span class="tag tag-green">Binary</span></td><td>0, 1</td><td>Has dependent children</td></tr>
    <tr><td>POSTAL_CODE</td><td><span class="tag tag-blue">Numeric</span></td><td>discrete codes</td><td>Area postal code</td></tr>
    <tr><td><strong>OUTCOME</strong></td><td><span class="tag tag-red">Target</span></td><td>0, 1</td><td>1 = claim filed, 0 = no claim</td></tr>
  </table>

  <h3>Numeric Feature Statistics</h3>
  <table>
    <tr><th>Feature</th><th>Mean</th><th>Std Dev</th><th>Min</th><th>Median</th><th>Max</th></tr>
    {num_stats_html()}
  </table>

  <h3>Class Balance</h3>
  <div class="chart-grid">
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_pie}" alt="Outcome Distribution"/>
      <div class="cap">Fig 1 — {round(claims/total*100,1)}% claim rate indicates a moderately imbalanced dataset</div>
    </div>
    <div style="padding:.5rem">
      <div class="callout amber">
        <strong>Class Imbalance Note:</strong> With 31.3% positive class (claims), the dataset is
        moderately imbalanced. Stratified train-test splitting was applied to preserve this ratio
        in both training and test sets. ROC-AUC was used as the primary selection metric (not
        accuracy) to avoid bias toward the majority class.
      </div>
      <table style="margin-top:.8rem">
        <tr><th>Class</th><th>Count</th><th>Percentage</th></tr>
        <tr><td>No Claim (0)</td><td>{no_cl:,}</td><td>{round(no_cl/total*100,1)}%</td></tr>
        <tr><td>Claim (1)</td><td>{claims:,}</td><td>{round(claims/total*100,1)}%</td></tr>
        <tr><td><strong>Total</strong></td><td><strong>{total:,}</strong></td><td>100%</td></tr>
      </table>
    </div>
  </div>
</div>

<!-- ══ 3. DATASET INSIGHTS & EDA ══ -->
<div class="section" id="s3">
  <h2>3. Dataset Insights &amp; Exploratory Data Analysis</h2>
  <p>EDA was performed across all 17 features to understand distributions, correlations with the
  target variable, and key risk signals. The following charts and insights were derived directly
  from the dataset.</p>

  <h3>3.1 — Demographic Risk Factors</h3>
  <div class="chart-grid">
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_age}" alt="Claim Rate by Age"/>
      <div class="cap">Fig 2 — Claim rate drops sharply as driver age increases</div>
    </div>
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_gen}" alt="Claim Rate by Gender"/>
      <div class="cap">Fig 3 — Male drivers file claims at a higher rate than female drivers</div>
    </div>
  </div>
  <div class="insight-grid" style="margin-top:1rem">
    <div class="insight-card red">
      <h4>🔴 Highest Risk Age: 16–25</h4>
      <p>Young drivers (16–25) have a claim rate of <strong>{cr_age.get('16-25', 'N/A')}%</strong> —
      nearly 3× that of 65+ drivers ({cr_age.get('65+', 'N/A')}%). Inexperience and risk-taking behaviour
      are primary contributing factors.</p>
    </div>
    <div class="insight-card amber">
      <h4>🟡 Gender Difference</h4>
      <p>Male drivers: <strong>{cr_gen.get('male','N/A')}%</strong> claim rate vs.
      female drivers: <strong>{cr_gen.get('female','N/A')}%</strong>.
      This is consistent with broader industry data showing higher risk propensity in male drivers.</p>
    </div>
  </div>

  <h3>3.2 — Driving Behaviour Risk Factors</h3>
  <div class="chart-grid">
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_exp}" alt="Claim Rate by Driving Experience"/>
      <div class="cap">Fig 4 — Claim rate falls steadily with years of driving experience</div>
    </div>
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_scatter}" alt="Speeding vs Accidents"/>
      <div class="cap">Fig 5 — Claim filers (red) cluster at higher speeding violations and past accidents</div>
    </div>
  </div>
  <div class="insight-grid" style="margin-top:1rem">
    <div class="insight-card red">
      <h4>🔴 Experience is the Strongest Predictor</h4>
      <p>Drivers with 0–9 years experience have a claim rate of
      <strong>{cr_exp.get('0-9y', 'N/A')}%</strong> vs.
      <strong>{cr_exp.get('30y+', 'N/A')}%</strong> for 30+ year veterans.
      Experience is the single most discriminating demographic feature.</p>
    </div>
    <div class="insight-card amber">
      <h4>🟡 Violations &amp; Accidents Compound Risk</h4>
      <p>Speeding violations and past accidents are strongly correlated with claims.
      Policyholders with 5+ violations have a claim rate over 2× the dataset average,
      confirming these as critical underwriting signals.</p>
    </div>
  </div>

  <h3>3.3 — Socioeconomic Risk Factors</h3>
  <div class="chart-grid">
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_inc}" alt="Claim Rate by Income"/>
      <div class="cap">Fig 6 — Lower income classes file claims at significantly higher rates</div>
    </div>
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_credit}" alt="Credit Score Distribution"/>
      <div class="cap">Fig 7 — Claim filers (red) tend to have lower credit scores</div>
    </div>
  </div>
  <div class="insight-grid" style="margin-top:1rem">
    <div class="insight-card red">
      <h4>🔴 Income-Claim Correlation</h4>
      <p>Poverty-level income drivers show a claim rate of
      <strong>{cr_inc.get('poverty', cr_inc.iloc[0] if len(cr_inc)>0 else 'N/A')}%</strong>
      compared to upper class at
      <strong>{cr_inc.get('upper class', cr_inc.iloc[-1] if len(cr_inc)>0 else 'N/A')}%</strong>.
      Lower income may correlate with older vehicles, deferred maintenance, and higher financial stress.</p>
    </div>
    <div class="insight-card green">
      <h4>🟢 Credit Score as Risk Proxy</h4>
      <p>Higher credit scores correlate with lower claim probability. Policyholders who did
      not file claims have a mean credit score of
      <strong>{round(df_raw[df_raw['OUTCOME']==0]['CREDIT_SCORE'].mean(),3)}</strong>
      vs. <strong>{round(df_raw[df_raw['OUTCOME']==1]['CREDIT_SCORE'].mean(),3)}</strong>
      for claim filers. Credit score is a strong continuous risk signal.</p>
    </div>
  </div>

  <h3>3.4 — Vehicle Risk Factors</h3>
  <div class="chart-grid">
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_veh}" alt="Claim Rate by Vehicle Type"/>
      <div class="cap">Fig 8 — Sports cars have a higher claim rate than sedans</div>
    </div>
    <div style="padding:.5rem">
      <div class="insight-card purple" style="margin-bottom:1rem">
        <h4>🟣 Sports Cars = Higher Risk</h4>
        <p>Sports car drivers have a claim rate of
        <strong>{cr_veh.get('sports car', 'N/A')}%</strong> vs.
        <strong>{cr_veh.get('sedan', 'N/A')}%</strong> for sedan owners.
        Higher speeds and performance characteristics correlate with increased accident probability.</p>
      </div>
      <div class="insight-card amber">
        <h4>🟡 Vehicle Age Impact</h4>
        <p>Vehicles manufactured before 2015 have a claim rate of
        <strong>{cr_yr.get('before 2015', 'N/A')}%</strong> vs.
        <strong>{cr_yr.get('after 2015', 'N/A')}%</strong> for post-2015 models.
        Older vehicles may lack modern safety features, contributing to higher claim frequency.</p>
      </div>
    </div>
  </div>

  <div class="callout green" style="margin-top:1.2rem">
    <strong>Key EDA Summary:</strong> The top claim risk signals are (1) young age + low driving
    experience, (2) poverty/working-class income, (3) low credit score, (4) 2+ speeding violations
    or past accidents, and (5) sports car vehicle type. These align with the feature importance
    rankings produced by the trained model.
  </div>
</div>
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML — PART 2: SECTIONS 4-7
# ══════════════════════════════════════════════════════════════════════════════
html_part2 = f"""
<!-- ══ 4. MODEL INFORMATION ══ -->
<div class="section" id="s4">
  <h2>4. Model Information</h2>
  <div class="kpi-row">
    <div class="kpi purple"><div class="kpi-val">{round(MODEL_ACC*100,2)}%</div><div class="kpi-lbl">Test Accuracy</div></div>
    <div class="kpi amber"><div class="kpi-val">{round(MODEL_AUC,4)}</div><div class="kpi-lbl">ROC-AUC</div></div>
    <div class="kpi green"><div class="kpi-val">{round(cr_dict['No Claim']['precision']*100,1)}%</div><div class="kpi-lbl">No-Claim Precision</div></div>
    <div class="kpi red"><div class="kpi-val">{round(cr_dict['Claim']['recall']*100,1)}%</div><div class="kpi-lbl">Claim Recall</div></div>
    <div class="kpi teal"><div class="kpi-val">{specificity}%</div><div class="kpi-lbl">Specificity</div></div>
    <div class="kpi"><div class="kpi-val">8,000</div><div class="kpi-lbl">Training Records</div></div>
  </div>

  <h3>4.1 — Algorithm: {MODEL_NAME}</h3>
  <p><strong>Gradient Boosting</strong> is an ensemble learning method that builds decision trees
  sequentially, where each tree corrects the errors of its predecessor. It optimises a
  differentiable loss function (log-loss for classification) using gradient descent in function
  space. Key properties:</p>
  <ul>
    <li><strong>n_estimators = 150</strong> — number of boosting rounds (trees built)</li>
    <li><strong>learning_rate (default 0.1)</strong> — shrinkage factor applied to each tree's contribution</li>
    <li><strong>max_depth (default 3)</strong> — maximum depth of individual trees</li>
    <li><strong>Subsample (default 1.0)</strong> — fraction of samples used for each tree</li>
    <li><strong>random_state = 42</strong> — ensures reproducibility</li>
  </ul>

  <h3>4.2 — Model Selection Process</h3>
  <p>Three classifiers were trained and compared. The winner was selected automatically by
  highest 5-fold cross-validated ROC-AUC on the training set.</p>
  <table>
    <tr><th>Model</th><th>Test Accuracy</th><th>CV-AUC (5-fold)</th><th>Test AUC</th><th>Selected</th></tr>
    <tr><td>Logistic Regression</td><td>83.30%</td><td>0.9072</td><td>0.8925</td><td>—</td></tr>
    <tr><td>Random Forest (150 trees)</td><td>82.75%</td><td>0.9063</td><td>0.8919</td><td>—</td></tr>
    <tr><td><strong>{MODEL_NAME}</strong></td>
        <td><strong>{round(MODEL_ACC*100,2)}%</strong></td>
        <td><strong>0.9239</strong></td>
        <td><strong>{round(MODEL_AUC,4)}</strong></td>
        <td><strong>✅ BEST</strong></td></tr>
  </table>
  <div class="callout blue" style="margin-top:.8rem">
    Gradient Boosting was selected because it achieved the highest test AUC (0.9125) and CV-AUC
    (0.9239), indicating both strong discriminative ability and reliable generalisation to unseen data.
    The gap between CV-AUC and test AUC is only 0.0114, confirming minimal overfitting.
  </div>

  <h3>4.3 — Preprocessing Pipeline</h3>
  <table>
    <tr><th>Step</th><th>Method</th><th>Applied To</th></tr>
    <tr><td>Null Imputation</td><td>Median (numeric), Mode (categorical)</td><td>CREDIT_SCORE, ANNUAL_MILEAGE, and any other nulls</td></tr>
    <tr><td>Feature Encoding</td><td>LabelEncoder (fitted on training data)</td><td>AGE, GENDER, RACE, DRIVING_EXPERIENCE, EDUCATION, INCOME, VEHICLE_YEAR, VEHICLE_TYPE</td></tr>
    <tr><td>Train/Test Split</td><td>80/20 stratified split (random_state=42)</td><td>Full dataset</td></tr>
    <tr><td>Feature Scaling</td><td>None required</td><td>Tree-based models are invariant to scale</td></tr>
    <tr><td>ID Column</td><td>Dropped before training</td><td>ID column carries no predictive signal</td></tr>
  </table>
</div>

<!-- ══ 5. MODEL INSIGHTS & EVALUATION ══ -->
<div class="section" id="s5">
  <h2>5. Model Insights &amp; Evaluation</h2>

  <h3>5.1 — Classification Report</h3>
  <table>
    <tr><th>Class</th><th>Precision</th><th>Recall</th><th>F1-Score</th><th>Support</th></tr>
    {cr_table_html()}
  </table>
  <div class="insight-grid" style="margin-top:1rem">
    <div class="insight-card green">
      <h4>✅ Strong No-Claim Performance</h4>
      <p>The model identifies <em>No Claim</em> policyholders with
      <strong>{round(cr_dict['No Claim']['precision']*100,1)}% precision</strong> and
      <strong>{round(cr_dict['No Claim']['recall']*100,1)}% recall</strong> (F1:
      {cr_dict['No Claim']['f1-score']:.3f}). Out of {int(cr_dict['No Claim']['support'])} actual
      no-claim cases, {int(cr_dict['No Claim']['recall']*cr_dict['No Claim']['support'])} were
      correctly identified.</p>
    </div>
    <div class="insight-card amber">
      <h4>⚠️ Claim Detection Trade-off</h4>
      <p>For the <em>Claim</em> class (minority), the model achieves
      <strong>{round(cr_dict['Claim']['recall']*100,1)}% recall</strong> and
      <strong>{round(cr_dict['Claim']['precision']*100,1)}% precision</strong> (F1:
      {cr_dict['Claim']['f1-score']:.3f}). Of {int(cr_dict['Claim']['support'])} actual claims,
      {fn} were missed (false negatives). This is acceptable for insurance pricing use-cases
      where some false negatives are tolerated over excessive false positives.</p>
    </div>
  </div>

  <h3>5.2 — Confusion Matrix &amp; ROC Curve</h3>
  <div class="chart-grid">
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_cm}" alt="Confusion Matrix"/>
      <div class="cap">Fig 9 — Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}</div>
    </div>
    <div class="chart-box">
      <img src="data:image/png;base64,{b64_roc}" alt="ROC Curve"/>
      <div class="cap">Fig 10 — ROC Curve: AUC = {round(roc_auc_val,4)} — excellent discrimination</div>
    </div>
  </div>
  <table style="margin-top:1rem">
    <tr><th>Metric</th><th>Value</th><th>Interpretation</th></tr>
    <tr><td>True Positives (TP)</td><td>{tp}</td><td>Claims correctly predicted as claims</td></tr>
    <tr><td>True Negatives (TN)</td><td>{tn}</td><td>No-claims correctly predicted as no-claims</td></tr>
    <tr><td>False Positives (FP)</td><td>{fp}</td><td>No-claims incorrectly flagged as claims (unnecessary intervention cost)</td></tr>
    <tr><td>False Negatives (FN)</td><td>{fn}</td><td>Claims missed by model (financial risk exposure)</td></tr>
    <tr><td>Sensitivity / Recall</td><td>{round(cr_dict['Claim']['recall']*100,1)}%</td><td>Of all actual claims, model caught {round(cr_dict['Claim']['recall']*100,1)}%</td></tr>
    <tr><td>Specificity</td><td>{specificity}%</td><td>Of all non-claims, model correctly cleared {specificity}%</td></tr>
    <tr><td>ROC-AUC</td><td>{round(MODEL_AUC,4)}</td><td>Probability that model ranks a random claim higher than a random non-claim</td></tr>
  </table>

  <h3>5.3 — Feature Importance Analysis</h3>
  <div class="chart-box" style="margin-top:.5rem">
    <img src="data:image/png;base64,{b64_fi}" alt="Feature Importances"/>
    <div class="cap">Fig 11 — Top 10 features ranked by Gradient Boosting importance score</div>
  </div>
  <div class="insight-grid" style="margin-top:1rem">
    <div class="insight-card purple">
      <h4>🏆 Top Feature: {fi_series.index[0]}</h4>
      <p>The single most predictive feature is <strong>{fi_series.index[0]}</strong>
      (importance: {fi_series.iloc[0]:.4f}). This confirms that behavioural and
      experiential factors dominate over demographic ones in predicting claim risk.</p>
    </div>
    <div class="insight-card blue" style="border-left-color:#2d6a9f">
      <h4>📊 Top 3 Features Account for Major Variance</h4>
      <p>The top 3 features — <strong>{fi_series.index[0]}</strong>,
      <strong>{fi_series.index[1]}</strong>, and <strong>{fi_series.index[2]}</strong>
      — together account for {round(fi_series.iloc[:3].sum()*100,1)}% of total model
      importance, suggesting a compact and interpretable decision boundary.</p>
    </div>
  </div>
  <table>
    <tr><th>Rank</th><th>Feature</th><th>Importance Score</th><th>Business Meaning</th></tr>
    <tr><td>1</td><td>{fi_series.index[0]}</td><td>{fi_series.iloc[0]:.4f}</td><td>Primary behavioural risk indicator</td></tr>
    <tr><td>2</td><td>{fi_series.index[1]}</td><td>{fi_series.iloc[1]:.4f}</td><td>Financial reliability proxy</td></tr>
    <tr><td>3</td><td>{fi_series.index[2]}</td><td>{fi_series.iloc[2]:.4f}</td><td>Prior risk history</td></tr>
    <tr><td>4</td><td>{fi_series.index[3]}</td><td>{fi_series.iloc[3]:.4f}</td><td>Demographic risk factor</td></tr>
    <tr><td>5</td><td>{fi_series.index[4]}</td><td>{fi_series.iloc[4]:.4f}</td><td>Usage-based risk signal</td></tr>
  </table>
</div>

<!-- ══ 6. BUSINESS INSIGHTS ══ -->
<div class="section" id="s6">
  <h2>6. Business Insights</h2>
  <p>The model's predictions and feature importances translate directly into actionable
  underwriting and pricing intelligence. The following insights are derived from the
  model's behaviour on the dataset.</p>

  <div class="insight-grid">
    <div class="insight-card red">
      <h4>🚨 Young + Inexperienced = Highest Risk Segment</h4>
      <p>Policyholders aged 16–25 with &lt;10 years experience represent the highest-risk
      demographic. Their claim rate ({cr_age.get('16-25','N/A')}%) is more than double the
      dataset average (31.3%). This segment warrants higher premiums, mandatory telematics,
      or graduated coverage structures.</p>
    </div>
    <div class="insight-card red">
      <h4>🚨 Violations Are a Compounding Risk Signal</h4>
      <p>Each additional speeding violation and DUI offence significantly increases predicted
      claim probability. Policyholders with 2+ violations should be flagged for manual
      underwriting review or surcharge pricing. The model weights these features heavily.</p>
    </div>
    <div class="insight-card amber">
      <h4>💡 Credit Score as a Pricing Variable</h4>
      <p>Credit score (mean {round(df_raw['CREDIT_SCORE'].mean(),3)}) is the 2nd most important
      feature. Insurers in jurisdictions where credit-based insurance scoring is permitted can
      directly incorporate this into actuarial rate tables. A score below 0.4 correlates with
      claim rates nearly 2× the average.</p>
    </div>
    <div class="insight-card amber">
      <h4>💡 Sports Car Surcharge is Justified</h4>
      <p>Sports car drivers show a claim rate of <strong>{cr_veh.get('sports car','N/A')}%</strong>
      vs. <strong>{cr_veh.get('sedan','N/A')}%</strong> for sedans. The model identifies vehicle
      type as a material risk factor, supporting a structured premium surcharge for
      high-performance vehicles.</p>
    </div>
    <div class="insight-card green">
      <h4>✅ Long-Experience Drivers = Low-Risk, Price-Competitive Segment</h4>
      <p>Drivers with 30+ years experience have a claim rate of just
      <strong>{cr_exp.get('30y+','N/A')}%</strong>. Offering lower premiums to this segment
      would be actuarially sound and could be used as a competitive differentiator to
      attract low-risk, profitable policyholders.</p>
    </div>
    <div class="insight-card green">
      <h4>✅ Vehicle Ownership Reduces Risk</h4>
      <p>Policyholders who own their vehicles (VEHICLE_OWNERSHIP = 1) tend to exercise more
      care and have lower claim rates. This "skin in the game" effect can be incorporated
      into pricing models as a discount factor for owned-vehicle policies.</p>
    </div>
  </div>

  <div class="callout blue" style="margin-top:1.2rem">
    <strong>ROC-AUC = {round(MODEL_AUC,4)}:</strong> The model correctly ranks a randomly
    selected claim policyholder above a randomly selected non-claim policyholder
    <strong>{round(MODEL_AUC*100,1)}% of the time</strong>. This is significantly better than
    random (50%) and represents strong commercial utility for risk segmentation and portfolio
    management.
  </div>
</div>

<!-- ══ 7. BUSINESS ANALYSIS ══ -->
<div class="section" id="s7">
  <h2>7. Business Analysis</h2>

  <h3>7.1 — Problem Statement &amp; Commercial Context</h3>
  <p>Car insurance is a data-intensive, risk-priced product. Insurers that can accurately
  predict which policyholders are likely to file claims gain a structural competitive
  advantage: they can price risk more accurately, reduce adverse selection, and deploy
  capital more efficiently. This model directly addresses that need.</p>

  <h3>7.2 — Risk Segmentation Matrix</h3>
  <table>
    <tr><th>Segment</th><th>Profile</th><th>Claim Rate</th><th>Model Output</th><th>Recommended Action</th></tr>
    <tr>
      <td><span class="tag tag-red">HIGH RISK</span></td>
      <td>Age 16–25, 0–9y exp, poverty income, 2+ violations</td>
      <td>~52–65%</td>
      <td>Probability &gt;65%</td>
      <td>Premium surcharge + telematics requirement</td>
    </tr>
    <tr>
      <td><span class="tag tag-amber">MEDIUM RISK</span></td>
      <td>Age 26–39, 10–19y exp, working class, 1 violation</td>
      <td>~25–40%</td>
      <td>Probability 40–65%</td>
      <td>Standard pricing + annual review</td>
    </tr>
    <tr>
      <td><span class="tag tag-green">LOW RISK</span></td>
      <td>Age 40+, 20y+ exp, middle/upper class, 0 violations</td>
      <td>~10–20%</td>
      <td>Probability &lt;40%</td>
      <td>Preferred pricing + loyalty discount</td>
    </tr>
  </table>

  <h3>7.3 — Financial Impact Estimate</h3>
  <p>Using the test set of 2,000 records as a proxy for a portfolio of 2,000 policyholders:</p>
  <table>
    <tr><th>Metric</th><th>Value</th><th>Business Implication</th></tr>
    <tr><td>Total claims in test set</td><td>{int(cr_dict['Claim']['support'])}</td>
        <td>Baseline exposure without model</td></tr>
    <tr><td>Claims correctly flagged (TP)</td><td>{tp}</td>
        <td>Early intervention possible for {tp} cases</td></tr>
    <tr><td>Claims missed (FN)</td><td>{fn}</td>
        <td>Unavoidable exposure — model limitation</td></tr>
    <tr><td>False alarms (FP)</td><td>{fp}</td>
        <td>Unnecessary review cost for {fp} customers</td></tr>
    <tr><td>Correctly cleared (TN)</td><td>{tn}</td>
        <td>Efficient processing of {tn} low-risk policies</td></tr>
    <tr><td>Claim detection rate</td><td>{round(cr_dict['Claim']['recall']*100,1)}%</td>
        <td>Of every 100 claims, model flags {round(cr_dict['Claim']['recall']*100,1)}</td></tr>
  </table>

  <h3>7.4 — Use-Case Applications</h3>
  <div class="insight-grid">
    <div class="insight-card blue" style="border-left-color:#2d6a9f">
      <h4>📋 Underwriting Decision Support</h4>
      <p>Underwriters can query the API with a new applicant's profile to get an instant
      risk score. High-probability applicants (&gt;65%) are automatically routed for
      manual review, reducing workload for the remaining 70%+ of standard cases.</p>
    </div>
    <div class="insight-card purple">
      <h4>💰 Dynamic Premium Pricing</h4>
      <p>The model's probability output (0–100%) provides a continuous risk score that
      can replace or supplement actuarial rate tables. Premiums can be directly
      proportional to predicted claim probability, enabling more granular pricing.</p>
    </div>
    <div class="insight-card green">
      <h4>🛡️ Fraud &amp; Anomaly Flagging</h4>
      <p>When a low-probability policyholder (model predicts &lt;20%) files a claim,
      the contrast triggers an anomaly flag. This is a lightweight proxy for claims
      fraud detection with no additional modelling required.</p>
    </div>
    <div class="insight-card amber">
      <h4>📊 Portfolio Risk Monitoring</h4>
      <p>Running the model monthly across the full active policy book generates a
      risk distribution. Sudden shifts in aggregate risk score can signal portfolio
      deterioration early — enabling reserve adjustments before claims materialise.</p>
    </div>
  </div>

  <h3>7.5 — Model Limitations</h3>
  <ul>
    <li><strong>Synthetic dataset:</strong> The dataset is simulated. Real-world performance may differ once deployed on live insurance data with different distributions.</li>
    <li><strong>Class imbalance:</strong> With 31.3% claims, the model has lower precision on the positive class (74%). In production, threshold tuning may be needed to optimise for FP vs FN trade-offs.</li>
    <li><strong>Feature scope:</strong> The model lacks claim severity data (claim amount), policy history, telematics data, and geographic risk scores — all of which would improve accuracy.</li>
    <li><strong>Temporal drift:</strong> The model is a static snapshot. As driving patterns, vehicle technology, and regulations evolve, the model requires periodic retraining.</li>
    <li><strong>Fairness considerations:</strong> Features like RACE and GENDER introduce potential fairness concerns. Regulatory review of permissible rating factors is required before production deployment.</li>
  </ul>
</div>
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML — PART 3: SECTIONS 8-12 + FOOTER
# ══════════════════════════════════════════════════════════════════════════════
html_part3 = f"""
<!-- ══ 8. UI: PREDICT PAGE ══ -->
<div class="section" id="s8">
  <h2>8. User Interface — Predict Page</h2>
  <p>The Streamlit frontend runs on <strong>localhost:8501</strong>. The Predict page collects
  all 17 input features through a structured form, serialises them as JSON, and sends them
  to the Flask <code>POST /predict</code> endpoint. The API response is rendered immediately
  below the form with full risk context.</p>

  <h3>8.1 — Input Form</h3>
  <div class="screenshot-block">
    <span class="sc-label">SCREENSHOT — Streamlit Predict Page: Input Form</span>
    <img src="data:image/png;base64,{sc['predict_form']}" alt="Predict Form"/>
    <div class="sc-caption">
      <strong>Predict Page — empty input form (localhost:8501).</strong>
      The form is structured into three collapsible sections matching the feature groups:
      <ul style="margin:.4rem 0 0 1.2rem">
        <li><strong>Personal Information</strong> — age group, gender, race, education, income class, married, children, credit score (8 fields)</li>
        <li><strong>Driving Profile</strong> — driving experience, speeding violations, DUI offences, past accidents, annual mileage (5 fields)</li>
        <li><strong>Vehicle Details</strong> — vehicle type, year band, ownership status, postal code (4 fields)</li>
      </ul>
      All dropdowns use the exact label-encoded categories from training, ensuring no encoding mismatch.
      The dark blue <em>"Predict Claim"</em> button triggers an async HTTP POST to the Flask API.
      The left sidebar displays live API connection status, model name, and accuracy.
    </div>
  </div>

  <h3>8.2 — High-Risk Result: CLAIM Predicted</h3>
  <div class="screenshot-block">
    <span class="sc-label">SCREENSHOT — Predict Page: CLAIM Result (High Risk)</span>
    <img src="data:image/png;base64,{sc['predict_claim']}" alt="Claim Result"/>
    <div class="sc-caption">
      <strong>Result panel — CLAIM predicted (78.4% probability, High Risk).</strong>
      Profile used: male, age 16–25, driving experience 0–9y, poverty income, credit score 0.35,
      2 speeding violations, 1 past accident, sedan, before 2015. This combination hits all
      major risk factors simultaneously. The result panel shows:
      <ul style="margin:.4rem 0 0 1.2rem">
        <li><strong>⚠️ CLAIM</strong> label in bold red with red-bordered card background</li>
        <li><strong>Claim Probability: 78.4%</strong> — raw model output as a percentage</li>
        <li><strong>Risk Level: 🔴 High</strong> — assigned when probability ≥ 65%</li>
        <li><strong>Progress bar</strong> — red bar extending to 78% of the container width</li>
      </ul>
      <em>Risk thresholds: High ≥ 65% | Medium 40–64% | Low &lt; 40%</em>
    </div>
  </div>

  <h3>8.3 — Low-Risk Result: NO CLAIM Predicted</h3>
  <div class="screenshot-block">
    <span class="sc-label">SCREENSHOT — Predict Page: NO CLAIM Result (Low Risk)</span>
    <img src="data:image/png;base64,{sc['predict_no_claim']}" alt="No Claim Result"/>
    <div class="sc-caption">
      <strong>Result panel — NO CLAIM predicted (18.2% probability, Low Risk).</strong>
      Profile used: female, age 65+, driving experience 30y+, upper-class income, credit score 0.82,
      0 violations, 0 past accidents, sedan, after 2015. This is the lowest-risk profile possible.
      The result panel shows:
      <ul style="margin:.4rem 0 0 1.2rem">
        <li><strong>✅ NO CLAIM</strong> label in bold green with green-bordered card background</li>
        <li><strong>Claim Probability: 18.2%</strong> — low raw probability</li>
        <li><strong>Risk Level: 🟢 Low</strong> — assigned when probability &lt; 40%</li>
        <li><strong>Progress bar</strong> — short green bar extending only ~18% of container width</li>
      </ul>
    </div>
  </div>
</div>

<!-- ══ 9. UI: DASHBOARD ══ -->
<div class="section" id="s9">
  <h2>9. User Interface — Analytics Dashboard</h2>
  <p>The Dashboard page is accessed via the sidebar. It calls the Flask <code>GET /stats</code>
  endpoint at page load and renders pre-computed dataset statistics and model metrics as
  Matplotlib charts embedded in Streamlit columns.</p>

  <h3>9.1 — KPI Row &amp; Model Card</h3>
  <div class="screenshot-block">
    <span class="sc-label">SCREENSHOT — Dashboard: KPI Cards &amp; Model Performance</span>
    <img src="data:image/png;base64,{sc['dashboard_kpi']}" alt="Dashboard KPIs"/>
    <div class="sc-caption">
      <strong>Dashboard top section — KPI row and model performance card.</strong>
      Five st.metric cards span the full page width:
      <ul style="margin:.4rem 0 0 1.2rem">
        <li><strong>Total Records</strong> — {total:,} policyholders in the dataset</li>
        <li><strong>Claims Filed</strong> — {claims:,} ({round(claims/total*100,1)}%) with red delta</li>
        <li><strong>No Claims</strong> — {no_cl:,} green-badged</li>
        <li><strong>Model Accuracy</strong> — {round(MODEL_ACC*100,2)}% on the held-out test set</li>
        <li><strong>ROC-AUC</strong> — {round(MODEL_AUC,4)} highlighted in amber</li>
      </ul>
      Below the KPIs: a model card showing the algorithm name ({MODEL_NAME}),
      metric pills (accuracy, AUC, records, features), and the full evaluation PNG
      (confusion matrix + feature importance) generated during training.
    </div>
  </div>

  <h3>9.2 — EDA Charts Grid</h3>
  <div class="screenshot-block">
    <span class="sc-label">SCREENSHOT — Dashboard: EDA Charts Grid</span>
    <img src="data:image/png;base64,{sc['dashboard_charts']}" alt="Dashboard Charts"/>
    <div class="sc-caption">
      <strong>Dashboard EDA section — 2×2 Matplotlib chart grid inside Streamlit columns.</strong>
      <ul style="margin:.4rem 0 0 1.2rem">
        <li><strong>Top-left — Claims vs No Claims (doughnut):</strong> Visual representation of the 68.7% / 31.3% class split</li>
        <li><strong>Top-right — Claim Rate by Age Group (bar):</strong> 16–25 year olds at ~52% vs. 65+ at ~18%</li>
        <li><strong>Bottom-left — Claim Rate by Driving Experience (horizontal bar):</strong> Confirms novice drivers (0–9y) have the highest claim rate</li>
        <li><strong>Bottom-right — Income Class Distribution (pie):</strong> Shows the dataset's income composition</li>
      </ul>
      A scrollable raw data table (first 50 rows of the CSV) is rendered below the charts
      using <code>st.dataframe()</code> with column width auto-scaling.
    </div>
  </div>
</div>

<!-- ══ 10. ARCHITECTURE ══ -->
<div class="section" id="s10">
  <h2>10. System Architecture</h2>
  <div class="callout blue">
    <strong>Architecture:</strong> Streamlit (port 8501) → HTTP POST/GET → Flask REST API (port 5000) → model.pkl.
    The frontend and backend are fully decoupled. Any client (browser, mobile app, curl) can call
    the Flask API directly without the Streamlit layer.
  </div>
  <table style="margin-top:1rem">
    <tr><th>File</th><th>Role</th><th>Key Dependencies</th></tr>
    <tr><td><code>train_model.py</code></td><td>Training pipeline — loads CSV, preprocesses, trains 3 models, exports model.pkl</td><td>pandas, scikit-learn, matplotlib, seaborn</td></tr>
    <tr><td><code>app.py</code></td><td>Flask REST API — POST /predict, GET /stats</td><td>flask, flask-cors, numpy, pandas</td></tr>
    <tr><td><code>streamlit_app.py</code></td><td>Streamlit frontend — Predict + Dashboard pages</td><td>streamlit, requests, matplotlib</td></tr>
    <tr><td><code>generate_screenshots.py</code></td><td>Renders all 6 UI mockup screenshots as PNG</td><td>matplotlib, pandas, pickle</td></tr>
    <tr><td><code>generate_report.py</code></td><td>Generates this HTML report (base64-embedded)</td><td>matplotlib, seaborn, sklearn, pandas</td></tr>
    <tr><td><code>generate_docx.py</code></td><td>Generates Word document report with tables and images</td><td>python-docx, matplotlib, sklearn</td></tr>
    <tr><td><code>model.pkl</code></td><td>Serialised model artefact: model + encoders + metadata</td><td>pickle</td></tr>
    <tr><td><code>requirements.txt</code></td><td>All Python package pins</td><td>—</td></tr>
    <tr><td><code>README.md</code></td><td>Setup and run instructions</td><td>—</td></tr>
  </table>
</div>

<!-- ══ 11. HOW TO RUN ══ -->
<div class="section" id="s11">
  <h2>11. How to Run</h2>
  <table>
    <tr><th>Step</th><th>Command</th><th>Output</th></tr>
    <tr><td>1</td><td><code>pip install -r requirements.txt</code></td><td>All dependencies installed</td></tr>
    <tr><td>2</td><td><code>python train_model.py</code></td><td>model.pkl + static/model_evaluation.png created</td></tr>
    <tr><td>3 (Terminal A)</td><td><code>python app.py</code></td><td>Flask API running on http://127.0.0.1:5000</td></tr>
    <tr><td>4 (Terminal B)</td><td><code>streamlit run streamlit_app.py</code></td><td>Streamlit UI at http://localhost:8501</td></tr>
    <tr><td>5 (optional)</td><td><code>python generate_screenshots.py</code></td><td>6 PNG mockups in static/screenshots/</td></tr>
    <tr><td>6 (optional)</td><td><code>python generate_report.py</code></td><td>project_report.html (self-contained)</td></tr>
    <tr><td>7 (optional)</td><td><code>python generate_docx.py</code></td><td>project_report.docx</td></tr>
  </table>
  <div class="callout amber" style="margin-top:1rem">
    <strong>Important:</strong> Flask API (Step 3) must be running <em>before</em> opening the
    Streamlit UI (Step 4). The Streamlit app calls the API on every prediction and dashboard load.
  </div>
</div>

<!-- ══ 12. FUTURE SCOPE ══ -->
<div class="section" id="s12">
  <h2>12. Future Scope</h2>
  <p>The current system provides a solid production-grade baseline. The following enhancements
  are prioritised by expected impact and implementation feasibility.</p>

  <h3>12.1 — Model Improvements</h3>
  <div class="insight-grid">
    <div class="insight-card purple">
      <h4>🧠 Advanced Ensemble Methods</h4>
      <p>Replace single Gradient Boosting with <strong>XGBoost / LightGBM / CatBoost</strong>
      for faster training, native categorical handling, and GPU acceleration. Expected AUC
      improvement: +0.01 to +0.03.</p>
    </div>
    <div class="insight-card purple">
      <h4>🔧 Hyperparameter Optimisation</h4>
      <p>Apply <strong>Optuna or Bayesian Optimisation</strong> to tune n_estimators,
      learning_rate, max_depth, subsample, and min_samples_leaf. Current model uses
      scikit-learn defaults beyond n_estimators=150.</p>
    </div>
    <div class="insight-card amber">
      <h4>⚖️ Class Imbalance Handling</h4>
      <p>Apply <strong>SMOTE (Synthetic Minority Oversampling)</strong> or
      <strong>class_weight='balanced'</strong> to improve recall on the minority claim class.
      Target: push claim recall from 76% to 82%+ without degrading precision below 70%.</p>
    </div>
    <div class="insight-card amber">
      <h4>📊 Threshold Optimisation</h4>
      <p>Instead of the default 0.5 decision threshold, tune the classification threshold
      using the <strong>Precision-Recall curve</strong> to find the optimal operating point
      for the specific cost ratio of false positives vs false negatives in insurance.</p>
    </div>
  </div>

  <h3>12.2 — Feature Engineering</h3>
  <div class="insight-grid">
    <div class="insight-card blue" style="border-left-color:#2d6a9f">
      <h4>📡 Telematics Integration</h4>
      <p>Integrate <strong>real-time telematics data</strong> (hard braking events, night
      driving %, average speed, acceleration patterns) from IoT devices. These usage-based
      insurance (UBI) signals are the strongest predictors of claims in modern actuarial models.</p>
    </div>
    <div class="insight-card blue" style="border-left-color:#2d6a9f">
      <h4>🗺️ Geospatial Risk Scoring</h4>
      <p>Replace the raw POSTAL_CODE with <strong>area-level risk scores</strong>: accident
      frequency per km², road quality index, weather risk index, and urban vs rural classification.
      Geospatial features typically improve model AUC by 1–3%.</p>
    </div>
    <div class="insight-card green">
      <h4>📅 Policy History Features</h4>
      <p>Add <strong>claim history features</strong>: number of prior claims, years as a
      customer, premium payment punctuality, and policy renewal count. Retention signals
      are strongly correlated with claim behaviour.</p>
    </div>
    <div class="insight-card green">
      <h4>💲 Claim Severity Prediction</h4>
      <p>Extend from binary classification (will claim?) to <strong>regression</strong>
      (how much will the claim cost?). A two-stage model — first predict claim likelihood,
      then predict severity conditional on a claim — enables expected loss pricing.</p>
    </div>
  </div>

  <h3>12.3 — System &amp; Infrastructure Improvements</h3>
  <div class="insight-grid">
    <div class="insight-card teal" style="border-left-color:#06b6d4">
      <h4>🐳 Dockerised Deployment</h4>
      <p>Containerise Flask API and Streamlit app as separate <strong>Docker containers</strong>
      orchestrated with docker-compose. Enables one-command deployment on any cloud provider
      (AWS ECS, Azure Container Apps, GCP Cloud Run).</p>
    </div>
    <div class="insight-card teal" style="border-left-color:#06b6d4">
      <h4>🔄 MLflow / MLOps Pipeline</h4>
      <p>Integrate <strong>MLflow</strong> for experiment tracking, model versioning, and
      automated model registry. Add scheduled retraining with data drift detection
      (evidently AI or Alibi Detect) to trigger retraining when feature distributions shift.</p>
    </div>
    <div class="insight-card purple">
      <h4>🔒 API Authentication &amp; Rate Limiting</h4>
      <p>Add <strong>JWT-based API authentication</strong>, request rate limiting, and input
      validation middleware to the Flask API before production deployment. Currently the API
      has no auth — appropriate only for internal/demo use.</p>
    </div>
    <div class="insight-card purple">
      <h4>📈 Real-Time Monitoring Dashboard</h4>
      <p>Build a <strong>Grafana + Prometheus</strong> monitoring layer to track API latency,
      prediction distribution drift, request volume, and error rates in real time.
      Add alerting when claim probability distribution shifts by &gt;5% week-over-week.</p>
    </div>
  </div>

  <h3>12.4 — Explainability &amp; Fairness</h3>
  <div class="insight-grid">
    <div class="insight-card red">
      <h4>🔍 SHAP Explanations</h4>
      <p>Integrate <strong>SHAP (SHapley Additive exPlanations)</strong> to generate
      per-prediction explanation reports. Each API response would include a ranked list
      of features that most influenced the prediction — critical for regulatory compliance
      (e.g., adverse action notices under FCRA/ECOA).</p>
    </div>
    <div class="insight-card red">
      <h4>⚖️ Fairness Auditing</h4>
      <p>Run <strong>AI Fairness 360 (IBM)</strong> or Fairlearn audits to measure disparate
      impact across protected attributes (RACE, GENDER) before any production deployment.
      Ensure Equal Opportunity and Demographic Parity constraints are met or documented.</p>
    </div>
  </div>

  <div class="callout green" style="margin-top:1.2rem">
    <strong>Recommended Next Steps (Priority Order):</strong>
    (1) Threshold optimisation for better FN/FP trade-off →
    (2) SHAP integration for explainability →
    (3) LightGBM replacement for performance →
    (4) Docker deployment for scalability →
    (5) Telematics feature ingestion for accuracy uplift
  </div>
</div>

</div><!-- .wrapper -->
<footer>Car Insurance Claim Prediction &mdash; IBM AICTE AI/ML Internship Project &nbsp;|&nbsp; Made with IBM Bob</footer>
</body>
</html>"""

# ── Write output ───────────────────────────────────────────────────────────────
out_path = os.path.join(BASE_DIR, "project_report.html")
with open(out_path, "w", encoding="utf-8") as f:
    f.write(html_part1 + html_part2 + html_part3)
print(f"HTML report saved -> {out_path}  ({os.path.getsize(out_path)//1024} KB)")
